# DSC LOGIKA UI 2026 — Credit Default Prediction

| Item | Detail |
|---|---|
| **Participant number** | DSC0152 |
| **Team** | kali ini beneran deh |
| **Members** | Agastya Tristan Dhaniswara · Dimaz Satya Ardendi · Jan Agra Adyuta Harnowo |
| **Metric** | Kolmogorov–Smirnov statistic |
| **Public score** | **0.56520** |

**Task.** Rank 30,000 test applications by the probability that the loan reaches
90+ days past due within 12 months of origination.

**Submitted model.** Six gradient-boosting legs and one spline logistic
regression, trained with 10-fold cross-validation repeated over six fold
partitions, then blended in rank space and extrapolated away from the
lean-column LightGBM leg:

```
trees4 = mean of rank(lgb_full, lgb_lean, cat_full, xgb_full)
core   = (2 * trees4 + lgbx_full) / 3
tm30   = 0.7 * core + 0.3 * linear
final  = rank01(tm30 + 0.75 * (tm30 - lgb_lean))
```

### How to run

Run all cells from top to bottom (*Run All*). Nothing has to be configured and
no file other than this notebook and the competition data is needed.

The notebook builds every feature from the raw CSVs, trains all seven legs,
blends them, and writes the submission. That takes about 6.5 hours on a
10-core CPU; the outputs saved in this notebook are from such a complete run.

The **data folder** is found automatically: `data/` next to or above the
notebook, any folder under `/kaggle/input` that holds `train_applications.csv`,
or a path given in the `DSC_DATA` environment variable.

### Inputs and outputs

| Item | Description |
|---|---|
| **Input** | the eight competition CSVs, nothing else |
| **Output** | `submission_DSC0152.csv` (`application_id,default_probability`, 30,000 rows) and `training_log.txt`, written next to this notebook |
| **Work folder** | the folder the notebook is run from; every generated file is written there |
| **Code** | all of it is in this notebook; sections 4 and 6 write the feature and training modules (`fe_base.py`, `fe_v65.py`, `fe_extra.py`, `trainer.py`) next to the notebook with `%%writefile` and import them |

### Environment used for the submission

Python 3.13.11 · numpy 2.5.1 · pandas 3.0.5 · scipy 1.18.0 · scikit-learn 1.9.0 ·
LightGBM 4.7.0 · XGBoost 3.3.0 · CatBoost 1.2.10 · matplotlib 3.11.1 · CPU only
(Apple M4, 10 cores).

### Contents

| section | what it does |
|---|---|
| 1. Setup | libraries, switches, data location, run mode |
| 2. Data | reads the tables and prints their basic facts |
| 3. Cutoff audit | proves no record is dated after `decision_datetime` |
| 4. Feature code | writes the three hand-written feature modules |
| 5. Feature frame | assembles the 598-column frame and the three column sets |
| 6. Models | the seven-leg trainer, or the stored predictions of its run |
| 7. Blend and submission | rank blend, out-of-fold scores, submission file, KS plot |
| 8. How the recipe was chosen | why every setting of the blend is the one it is |
| 9. Submission validation | format checks and the byte-identity check |
| 10. Final summary | one table with the result of the run |
| 11. Limits | what this notebook does not claim |

## 1. Setup

### 1.1 Libraries and switches

There is nothing to switch. `R_FINAL` is the extrapolation ratio of the
submitted blend, and `SUBMITTED_SHA256` is the fingerprint of the CSV that was submitted to Kaggle and
scored 0.56520; section 9 compares the rebuilt file against it.

In [ ]:
import hashlib
import os
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import rankdata, spearmanr
from sklearn.metrics import roc_auc_score

R_FINAL = 0.75         # extrapolation ratio of the submitted vector
SUBMITTED_SHA256 = "35260db3730a3eec7bc2cce9fb3b540d047c64d5d980e39ab19b07ebf07c733b"

### 1.2 Locate the data and choose the run mode

Searches for the data folder and uses the notebook's own folder as the work
folder. The printed versions are the ones this run used.

In [ ]:
def find_data():
    """Folder holding the competition CSVs."""
    here = Path.cwd().absolute()
    candidates = [Path(os.environ["DSC_DATA"])] if "DSC_DATA" in os.environ else []
    candidates += [p / "data" for p in [here, *here.parents]]
    kaggle = Path("/kaggle/input")
    if kaggle.exists():
        candidates += [p.parent for p in kaggle.rglob("train_applications.csv")]
    for c in candidates:
        if (c / "train_applications.csv").exists():
            return c.absolute()
    raise FileNotFoundError("train_applications.csv not found; set DSC_DATA to the data folder")


DATA = find_data()
WORK = Path.cwd().absolute()                 # the notebook's own folder: modules and outputs go here
sys.path.insert(0, str(WORK))

import sklearn, lightgbm, xgboost, catboost
print(f"data folder   : {DATA}")
print(f"work folder   : {WORK}")
print(f"python {sys.version.split()[0]} | numpy {np.__version__} | pandas {pd.__version__} | "
      f"sklearn {sklearn.__version__}")
print(f"lightgbm {lightgbm.__version__} | xgboost {xgboost.__version__} | catboost {catboost.__version__}")

## 2. Data

The eight provided CSVs and nothing else. No external data is read anywhere in
this notebook.

| table | grain | role |
|---|---|---|
| `train_applications` / `test_applications` | one row per application | the rows to score; `target` = reached 90+ DPD within 12 months |
| `customer_profiles` | one row per customer | demographics and declared income |
| `previous_applications` | many per customer | earlier loan applications and their outcomes |
| `repayment_history` | many per customer | instalment due dates, payments and days past due |
| `account_behavior` | one per customer-month | balances, inflows and outflows |
| `bureau_observations` | many per customer | credit-bureau snapshots with an availability date |
| `ecommerce_transactions` | many per customer | e-commerce purchases |

### 2.1 Load the applications and print basic facts

Counts every table, loads the two application tables, and checks the facts the
model relies on: the default rate, that no customer appears in both train and
test, and that every loan has a 12-month term.

In [ ]:
TABLES = ["train_applications", "test_applications", "customer_profiles",
          "previous_applications", "repayment_history", "account_behavior",
          "bureau_observations", "ecommerce_transactions"]
shapes = {}
for name in TABLES:
    n = sum(1 for _ in open(DATA / f"{name}.csv")) - 1
    cols = len(pd.read_csv(DATA / f"{name}.csv", nrows=1).columns)
    shapes[name] = (n, cols)

train = pd.read_csv(DATA / "train_applications.csv",
                    parse_dates=["application_datetime", "decision_datetime"])
test = pd.read_csv(DATA / "test_applications.csv",
                   parse_dates=["application_datetime", "decision_datetime"])

print(pd.DataFrame(shapes, index=["rows", "cols"]).T.to_string())
print(f"\ntrain {len(train):,} rows | test {len(test):,} rows "
      f"| positive rate {train.target.mean():.6f}")
print(f"customers appearing in both sets: "
      f"{len(set(train.customer_id) & set(test.customer_id))}")
print(f"term_months values   : {sorted(int(v) for v in train.term_months.unique())}")
print(f"interest rate values : {sorted(float(v) for v in train.annual_interest_rate.unique())}")

## 3. Cutoff audit

Every feature may use only information available at the selected application's
`decision_datetime`. Two things are checked: whether any provided table holds a
record dated after that cutoff, and whether the bureau observation attached to
the application was actually available at decision time.

### 3.1 Count records dated after the cutoff

Joins every event table to the decision time of its customer's application and
counts events that happen later. An account month counts as known only once it
has closed. The cell stops the notebook if any count is above zero.

In [ ]:
apps = pd.concat([train[["application_id", "customer_id", "decision_datetime"]],
                  test[["application_id", "customer_id", "decision_datetime"]]])
cut = apps[["customer_id", "decision_datetime"]].rename(columns={"decision_datetime": "cut"})
audit = {}

pa = pd.read_csv(DATA / "previous_applications.csv", parse_dates=["decision_datetime"]).merge(cut, on="customer_id")
audit["previous_applications.decision_datetime"] = (len(pa), int((pa.decision_datetime > pa.cut).sum()))
rh = pd.read_csv(DATA / "repayment_history.csv", parse_dates=["due_date", "last_payment_datetime"]).merge(cut, on="customer_id")
audit["repayment_history.due_date"] = (len(rh), int((rh.due_date > rh.cut).sum()))
audit["repayment_history.last_payment_datetime"] = (len(rh), int((rh.last_payment_datetime > rh.cut).sum()))
ab = pd.read_csv(DATA / "account_behavior.csv", parse_dates=["month"]).merge(cut, on="customer_id")
audit["account_behavior.month_closed"] = (len(ab), int(((ab.month + pd.offsets.MonthBegin(1)) > ab.cut).sum()))
ec = pd.read_csv(DATA / "ecommerce_transactions.csv", parse_dates=["transaction_datetime"]).merge(cut, on="customer_id")
audit["ecommerce_transactions.transaction_datetime"] = (len(ec), int((ec.transaction_datetime > ec.cut).sum()))
bu = pd.read_csv(DATA / "bureau_observations.csv", parse_dates=["report_datetime", "available_datetime"])
sel = bu.merge(apps, on=["application_id", "customer_id"], how="inner")
audit["bureau_observations.available_datetime"] = (len(sel), int((sel.available_datetime > sel.decision_datetime).sum()))
audit["bureau_observations.report_datetime"] = (len(sel), int((sel.report_datetime > sel.decision_datetime).sum()))

rows = pd.DataFrame([(k, v[0], v[1]) for k, v in audit.items()],
                    columns=["source field", "rows joined", "rows after cutoff"])
print(rows.to_string(index=False))
post = int(rows["rows after cutoff"].sum())
print(f"\ntotal records dated after the cutoff: {post}")
assert post == 0, "a source holds post-cutoff records; window features need a positive-time filter"
print("PASS: the provided tables are already truncated at decision_datetime")
del pa, rh, ab, ec, bu, sel

## 4. Feature code

Three hand-written modules build one row per application. Every historical
aggregate is computed from `days_ago = decision_datetime - event` and windowed
over that quantity, so nothing dated after the decision can enter a feature.

| module | content |
|---|---|
| `fe_base.py` | application terms, customer profile, previous applications, repayment history, account behavior, bureau observations, e-commerce behavior, ratio and trend interactions |
| `fe_v65.py` | data-quality (missingness and coverage) and cash-flow stress blocks |
| `fe_extra.py` | second-round blocks: exponentially weighted repayment and bureau history, within-grade z-scores, account and e-commerce extras, cross-source consistency, interactions |

The cells below write the modules next to the notebook and do not compute anything yet. 

### 4.1 `fe_base.py` — base features (311 columns)

Entry point `build()`. One block per source table, each returning customer-level
aggregates over time windows measured back from the decision date (from 7 days
up to 800 days), followed by debt-to-income and cash-flow ratios that combine blocks.
`factor_inputs` / `factor_features` at the end build fold-safe factor encodings,
used by the trainer in section 6.

In [ ]:
%%writefile fe_base.py
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

D = "data/"


def load_apps():
    dt = ["application_datetime", "decision_datetime"]
    tr = pd.read_csv(D + "train_applications.csv", parse_dates=dt)
    te = pd.read_csv(D + "test_applications.csv", parse_dates=dt)
    return pd.concat([tr, te], ignore_index=True)


def app_features(app):
    f = app[["application_id", "customer_id", "decision_datetime"]].copy()
    f["loan_amount"] = app.loan_amount
    f["interest_rate"] = app.annual_interest_rate
    f["installment"] = app.proposed_monthly_installment
    f["loan_purpose"] = app.loan_purpose.astype("category")
    f["decision_lag_h"] = (app.decision_datetime - app.application_datetime).dt.total_seconds() / 3600
    f["app_hour"] = app.application_datetime.dt.hour
    f["app_dow"] = app.application_datetime.dt.dayofweek
    # default rates move with the season (12% in May, 15% in February), so the decision date enters as day of year
    f["dec_doy"] = app.decision_datetime.dt.dayofyear
    return f


def profile_features(f):
    cp = pd.read_csv(D + "customer_profiles.csv")
    for c in ["region", "employment_type", "housing_status"]:
        cp[c] = cp[c].astype("category")
    f = f.merge(cp, on="customer_id", how="left")
    inc = f.declared_monthly_income
    f["inst_to_income"] = f.installment / inc
    f["loan_to_income"] = f.loan_amount / inc
    f["income_per_dependent"] = inc / (f.dependents + 1)
    f["tenure_to_age"] = f.employment_tenure_months / (f.age * 12)
    return f


def bureau_features(app):
    bu = pd.read_csv(D + "bureau_observations.csv", parse_dates=["report_datetime", "available_datetime"])
    bu = bu.merge(app[["customer_id", "application_id", "decision_datetime"]], on="customer_id", suffixes=("", "_sel"))
    bu["age_days"] = (bu.decision_datetime - bu.report_datetime).dt.total_seconds() / 86400
    bu["unscored"] = bu.score.isna().astype(int)
    bu["revolving_free"] = bu.revolving_credit_limit - bu.revolving_outstanding_balance

    cur = bu[bu.application_id == bu.application_id_sel]
    g = cur.groupby("application_id_sel")
    out = g.agg(
        bu_score_mean=("score", "mean"), bu_score_min=("score", "min"), bu_score_max=("score", "max"),
        bu_score_std=("score", "std"), bu_unscored=("unscored", "sum"), bu_reports=("score", "size"),
        bu_active_max=("active_accounts", "max"), bu_active_mean=("active_accounts", "mean"),
        bu_outstanding_max=("outstanding_balance", "max"), bu_outstanding_mean=("outstanding_balance", "mean"),
        bu_rev_limit_max=("revolving_credit_limit", "max"), bu_rev_bal_max=("revolving_outstanding_balance", "max"),
        bu_util_max=("utilization", "max"), bu_util_mean=("utilization", "mean"),
        bu_inq_max=("inquiries_last_6m", "max"), bu_inq_mean=("inquiries_last_6m", "mean"),
        bu_delinq_max=("delinquent_accounts_last_12m", "max"), bu_delinq_sum=("delinquent_accounts_last_12m", "sum"),
        bu_age_min=("age_days", "min"), bu_age_max=("age_days", "max"),
    )
    # the freshest report is the best snapshot of the customer at decision time
    latest = cur.sort_values("report_datetime").groupby("application_id_sel").last()
    for c in ["score", "outstanding_balance", "utilization", "inquiries_last_6m", "delinquent_accounts_last_12m", "active_accounts"]:
        out["bu_latest_" + c] = latest[c]
    piv = cur.pivot_table(index="application_id_sel", columns="bureau_provider", values="score")
    out = out.join(piv.add_prefix("bu_score_"))

    hist = bu[bu.application_id != bu.application_id_sel]
    h = hist.groupby("application_id_sel").agg(
        buh_score_mean=("score", "mean"), buh_score_min=("score", "min"),
        buh_delinq_max=("delinquent_accounts_last_12m", "max"), buh_outstanding_max=("outstanding_balance", "max"),
        buh_inq_max=("inquiries_last_6m", "max"), buh_n=("score", "size"),
    )
    hl = hist.sort_values("report_datetime").groupby("application_id_sel").last()
    h["buh_last_score"] = hl.score
    h["buh_last_age_days"] = hl.age_days
    out = out.join(h)
    out["bu_score_trend"] = out.bu_score_mean - out.buh_score_mean
    out["bu_score_trend_last"] = out.bu_latest_score - out.buh_last_score
    # every report is a noisy reading of the same customer, so pooling all of them reduces noise
    pool = ["score", "utilization", "inquiries_last_6m", "delinquent_accounts_last_12m", "outstanding_balance",
            "revolving_outstanding_balance", "active_accounts"]
    out = out.join(bu.groupby("application_id_sel")[pool].mean().add_prefix("bua_"))
    out["bua_score_median"] = bu.groupby("application_id_sel").score.median()
    out["bua_unscored_rate"] = bu.groupby("application_id_sel").unscored.mean()
    return out.rename_axis("application_id").reset_index()


def previous_features(app):
    pa = pd.read_csv(D + "previous_applications.csv", parse_dates=["application_datetime", "decision_datetime"])
    pa = pa.merge(app[["customer_id", "decision_datetime"]].rename(columns={"decision_datetime": "cut"}), on="customer_id")
    pa["days_ago"] = (pa.cut - pa.decision_datetime).dt.total_seconds() / 86400
    pa["approved"] = (pa.decision == "approved").astype(int)
    pa["rejected"] = (pa.decision == "rejected").astype(int)
    pa["cancelled"] = (pa.decision == "cancelled").astype(int)
    # months elapsed since origination tells whether the loan is still being repaid
    pa["months_elapsed"] = pa.days_ago / 30.44
    pa["active"] = (pa.approved == 1) & (pa.months_elapsed < pa.term_months)
    pa["active_inst"] = np.where(pa.active, pa.scheduled_monthly_installment, 0)
    pa["remaining_principal"] = np.where(
        pa.active, pa.approved_amount * (1 - pa.months_elapsed / pa.term_months), 0)
    for w in [90, 180, 365]:
        pa[f"in{w}"] = (pa.days_ago <= w).astype(int)
        pa[f"rej{w}"] = pa.rejected * pa[f"in{w}"]

    g = pa.groupby("customer_id")
    out = g.agg(
        pa_n=("decision", "size"), pa_approved=("approved", "sum"), pa_rejected=("rejected", "sum"),
        pa_cancelled=("cancelled", "sum"), pa_req_mean=("requested_amount", "mean"),
        pa_req_max=("requested_amount", "max"), pa_appr_sum=("approved_amount", "sum"),
        pa_rate_mean=("annual_interest_rate", "mean"), pa_rate_max=("annual_interest_rate", "max"),
        pa_term_mean=("term_months", "mean"), pa_last_days=("days_ago", "min"), pa_first_days=("days_ago", "max"),
        pa_active_n=("active", "sum"), pa_active_inst=("active_inst", "sum"),
        pa_remaining=("remaining_principal", "sum"),
        pa_in90=("in90", "sum"), pa_in180=("in180", "sum"), pa_in365=("in365", "sum"),
        pa_rej90=("rej90", "sum"), pa_rej365=("rej365", "sum"),
    )
    out["pa_reject_rate"] = out.pa_rejected / out.pa_n
    return out.reset_index()


def repayment_features(app):
    rh = pd.read_csv(D + "repayment_history.csv", parse_dates=["due_date", "last_payment_datetime"])
    rh = rh.merge(app[["customer_id", "decision_datetime"]].rename(columns={"decision_datetime": "cut"}), on="customer_id")
    rh["days_ago"] = (rh.cut - rh.due_date).dt.total_seconds() / 86400
    rh["unpaid"] = rh.amount_due - rh.amount_paid
    rh["paid_ratio"] = rh.amount_paid / rh.amount_due
    rh["pay_delay"] = (rh.last_payment_datetime - rh.due_date).dt.total_seconds() / 86400
    rh["late"] = (rh.max_days_past_due > 0).astype(int)
    rh["late30"] = (rh.max_days_past_due >= 30).astype(int)
    rh["late90"] = (rh.max_days_past_due >= 90).astype(int)
    rh["open_dpd"] = (rh.days_past_due > 0).astype(int)
    rh["recent"] = rh.days_ago <= 180
    for w in [90, 365]:
        rh[f"late_{w}"] = rh.late.where(rh.days_ago <= w)
        rh[f"dpd_{w}"] = rh.max_days_past_due.where(rh.days_ago <= w)
    for c in ["max_days_past_due", "late", "late30", "paid_ratio", "pay_delay"]:
        rh["r_" + c] = rh[c].where(rh.recent)
    # an installment due t days ago cannot show more than t days past due, so each rate only counts matured ones
    thresholds = [1, 15, 30, 60, 90]
    for t in thresholds:
        rh[f"mat{t}"] = (rh.days_ago >= t).astype(int)
        rh[f"hit{t}"] = ((rh.max_days_past_due >= t) & (rh.days_ago >= t)).astype(int)
    rh["sev"] = rh.max_days_past_due.where(rh.late == 1)
    rh["loan_seq"] = rh.groupby(["customer_id", "loan_id"]).due_date.rank()

    g = rh.groupby("customer_id")
    out = g.agg(
        rh_n=("amount_due", "size"), rh_loans=("loan_id", "nunique"),
        rh_dpd_now_max=("days_past_due", "max"), rh_dpd_now_sum=("days_past_due", "sum"),
        rh_open_dpd=("open_dpd", "sum"), rh_maxdpd_max=("max_days_past_due", "max"),
        rh_maxdpd_mean=("max_days_past_due", "mean"), rh_late=("late", "mean"), rh_late_n=("late", "sum"),
        rh_late30=("late30", "sum"), rh_late90=("late90", "sum"), rh_unpaid=("unpaid", "sum"),
        rh_paid_ratio=("paid_ratio", "mean"), rh_paid_ratio_min=("paid_ratio", "min"),
        rh_delay_mean=("pay_delay", "mean"), rh_delay_max=("pay_delay", "max"),
        rh_r_maxdpd=("r_max_days_past_due", "max"), rh_r_late=("r_late", "mean"),
        rh_r_late30=("r_late30", "sum"), rh_r_paid_ratio=("r_paid_ratio", "mean"),
        rh_r_delay=("r_pay_delay", "mean"), rh_last_due_days=("days_ago", "min"),
        rh_late_90=("late_90", "mean"), rh_late_365=("late_365", "mean"),
        rh_dpd_90=("dpd_90", "max"), rh_dpd_365=("dpd_365", "max"), rh_dpd_365_mean=("dpd_365", "mean"),
        rh_sev_mean=("sev", "mean"),
        **{f"rh_hit{t}": (f"hit{t}", "sum") for t in thresholds},
        **{f"rh_mat{t}": (f"mat{t}", "sum") for t in thresholds},
    )
    for t in thresholds:
        out[f"rh_rate{t}"] = out[f"rh_hit{t}"] / out[f"rh_mat{t}"].replace(0, np.nan)
        out[f"rh_rate{t}_smooth"] = (out[f"rh_hit{t}"] + 0.3) / (out[f"rh_mat{t}"] + 1)
    out = out.drop(columns=[f"rh_mat{t}" for t in thresholds])
    # worst loan and first installments of each loan, early delinquency is a strong warning sign
    per_loan = rh.groupby(["customer_id", "loan_id"]).agg(l_late=("late", "mean"), l_dpd=("max_days_past_due", "mean"), l_end=("due_date", "max"))
    out["rh_worst_loan_late"] = per_loan.groupby("customer_id").l_late.max()
    out["rh_best_loan_late"] = per_loan.groupby("customer_id").l_late.min()
    out["rh_last_loan_late"] = per_loan.sort_values("l_end").groupby("customer_id").l_late.last()
    out["rh_last_loan_dpd"] = per_loan.sort_values("l_end").groupby("customer_id").l_dpd.last()
    out["rh_early_late"] = rh[rh.loan_seq <= 3].groupby("customer_id").late.mean()
    last = rh.sort_values("due_date").groupby("customer_id").last()
    out["rh_last_dpd"] = last.days_past_due
    out["rh_last_paid_ratio"] = last.paid_ratio
    return out.reset_index()


def account_features(app):
    ab = pd.read_csv(D + "account_behavior.csv", parse_dates=["month"])
    ab = ab.merge(app[["customer_id", "decision_datetime"]], on="customer_id")
    dm = ab.decision_datetime.dt.year * 12 + ab.decision_datetime.dt.month
    ab["k"] = dm - (ab.month.dt.year * 12 + ab.month.dt.month)
    ab = ab.sort_values(["customer_id", "month"])
    ab["net"] = ab.inflows - ab.outflows
    ab["out_in"] = ab.outflows / ab.inflows.replace(0, np.nan)
    ab["rank"] = ab.groupby("customer_id").cumcount(ascending=False)
    cols = ["inflows", "outflows", "income_like_credits", "average_balance", "minimum_balance", "low_balance_days", "net", "out_in"]

    g = ab.groupby("customer_id")
    out = g[cols].mean().add_prefix("ab_mean_")
    out = out.join(ab[ab["rank"] < 3].groupby("customer_id")[cols].mean().add_prefix("ab_l3_"))
    out = out.join(ab[ab.k == 1].groupby("customer_id")[cols].mean().add_prefix("ab_k1_"))
    base = ab[(ab.k >= 4) & (ab.k <= 12)].groupby("customer_id")[cols].mean()
    for c in ["inflows", "outflows", "income_like_credits", "average_balance", "net"]:
        out[f"ab_k1_vs_base_{c}"] = out[f"ab_k1_{c}"] / base[c].abs().replace(0, np.nan)
        out[f"ab_l3_vs_base_{c}"] = out[f"ab_l3_{c}"] / base[c].abs().replace(0, np.nan)
    out["ab_last_k"] = g.k.min()
    out = out.join(g[["income_like_credits", "inflows", "average_balance"]].std().add_prefix("ab_std_"))
    out["ab_months"] = g.size()
    out["ab_low_days_max"] = g.low_balance_days.max()
    out["ab_low_months"] = g.low_balance_days.apply(lambda s: (s > 0).sum())
    out["ab_min_balance_min"] = g.minimum_balance.min()
    out["ab_last_close"] = g.closing_balance.last()
    out["ab_first_open"] = g.opening_balance.first()
    out["ab_balance_growth"] = out.ab_last_close - out.ab_first_open
    out["ab_inc_trend"] = out.ab_l3_income_like_credits - out.ab_mean_income_like_credits
    out["ab_bal_trend"] = out.ab_l3_average_balance - out.ab_mean_average_balance
    out["ab_zero_income_months"] = g.income_like_credits.apply(lambda s: (s == 0).sum())
    return out.reset_index()


def ecommerce_features(app):
    ec = pd.read_csv(D + "ecommerce_transactions.csv", parse_dates=["transaction_datetime"])
    ec = ec.merge(app[["customer_id", "decision_datetime"]].rename(columns={"decision_datetime": "cut"}), on="customer_id")
    ec["done"] = ec.order_status == "completed"
    ec["spend"] = ec.item_total.where(ec.done, 0)

    od = ec.drop_duplicates("order_id").copy()
    od["days_ago"] = (od.cut - od.transaction_datetime).dt.total_seconds() / 86400
    od["spend"] = od.order_total.where(od.done, 0)
    od["cancel"] = (~od.done).astype(int)
    windows = [7, 14, 30, 60, 90, 180, 365]
    for w in windows:
        od[f"spend{w}"] = od.spend.where(od.days_ago <= w, 0)
        od[f"n{w}"] = (od.days_ago <= w).astype(int)
    od["spend_base"] = od.spend.where((od.days_ago > 90) & (od.days_ago <= 365), 0)
    od["n_base"] = ((od.days_ago > 90) & (od.days_ago <= 365)).astype(int)
    for m in ["COD", "credit_card", "e_wallet", "bank_transfer", "debit_card"]:
        od["pm_" + m] = (od.payment_method == m).astype(int)

    g = od.groupby("customer_id")
    out = g.agg(
        ec_orders=("order_id", "size"), ec_spend=("spend", "sum"), ec_spend_mean=("spend", "mean"),
        ec_spend_max=("spend", "max"), ec_spend_base=("spend_base", "sum"), ec_n_base=("n_base", "sum"),
        **{f"ec_spend{w}": (f"spend{w}", "sum") for w in windows},
        **{f"ec_n{w}": (f"n{w}", "sum") for w in windows},
        ec_cancel_rate=("cancel", "mean"), ec_last_days=("days_ago", "min"), ec_first_days=("days_ago", "max"),
        ec_cod=("pm_COD", "mean"), ec_cc=("pm_credit_card", "mean"), ec_ewallet=("pm_e_wallet", "mean"),
        ec_bank=("pm_bank_transfer", "mean"), ec_debit=("pm_debit_card", "mean"),
    )
    out["ec_spend90_share"] = out.ec_spend90 / out.ec_spend.replace(0, np.nan)
    # recent monthly spend against the customer's own baseline month (days 91 to 365)
    base_month = out.ec_spend_base / 9 + 1
    base_n = out.ec_n_base / 9 + 0.1
    for w in [7, 14, 30, 60, 90]:
        out[f"ec_spend{w}_vs_base"] = out[f"ec_spend{w}"] * 30 / w / base_month
        out[f"ec_n{w}_vs_base"] = out[f"ec_n{w}"] * 30 / w / base_n
    recent = od[od.days_ago <= 90]
    out["ec_r_cancel_rate"] = recent.groupby("customer_id").cancel.mean()
    out["ec_r_cc"] = recent.groupby("customer_id").pm_credit_card.mean()
    out["ec_r_cod"] = recent.groupby("customer_id").pm_COD.mean()
    out["ec_r_order_max"] = recent.groupby("customer_id").spend.max()
    out["ec_r_item_mean"] = ec[ec.done & (ec.cut - ec.transaction_datetime).dt.days.le(90)].groupby("customer_id").item_total.mean()
    out["ec_item_mean"] = ec[ec.done].groupby("customer_id").item_total.mean()
    out["ec_item_mean_ratio"] = out.ec_r_item_mean / out.ec_item_mean
    # exponentially decayed activity, the half life controls how much weight the newest orders get
    for hl in [15, 45]:
        w = np.exp(-od.days_ago * np.log(2) / hl)
        out[f"ec_decay_n{hl}"] = w.groupby(od.customer_id).sum()
        out[f"ec_decay_spend{hl}"] = (w * od.spend).groupby(od.customer_id).sum()
        out[f"ec_decay_n{hl}_vs_base"] = out[f"ec_decay_n{hl}"] / base_n
    cat = ec[ec.done].pivot_table(index="customer_id", columns="product_category", values="spend", aggfunc="sum", fill_value=0)
    out = out.join(cat.div(cat.sum(axis=1), axis=0).add_prefix("ec_share_"))
    out = out.join(cat.add_prefix("ec_catspend_"))
    # unit price against the category median shows whether a customer is trading up beyond their usual basket
    it = ec[ec.quantity > 0].copy()
    unit = it.item_total / it.quantity
    it["lrel"] = np.log(unit / unit.groupby(it.product_category).transform("median"))
    it["days_ago"] = (it.cut - it.transaction_datetime).dt.days
    gi = it.groupby("customer_id")
    out["ec_price_rel"] = gi.lrel.mean()
    out["ec_price_rel_std"] = gi.lrel.std()
    out["ec_price_rel_max"] = gi.lrel.max()
    for lo, hi in [(0, 30), (0, 90), (90, 365), (365, 800)]:
        m = (it.days_ago > lo) & (it.days_ago <= hi)
        out[f"ec_price_rel_{lo}_{hi}"] = it[m].groupby("customer_id").lrel.mean()
    out["ec_price_rel_trend"] = out.ec_price_rel_0_90 - out.ec_price_rel_365_800
    out["ec_price_rel_trend2"] = out.ec_price_rel_0_90 - out.ec_price_rel_90_365
    slope = gi.apply(lambda d: np.polyfit(-d.days_ago, d.lrel, 1)[0] if len(d) > 2 else np.nan)
    out["ec_price_rel_slope"] = slope * 365
    out = out.join(it.pivot_table(index="customer_id", columns="product_category", values="lrel", aggfunc="mean").add_prefix("ec_price_rel_"))
    cnt = ec[ec.done].pivot_table(index="customer_id", columns="product_category", values="quantity", aggfunc="sum", fill_value=0)
    out = out.join(cnt.add_prefix("ec_catqty_"))
    return out.reset_index()


def sequence_features(app):
    rh = pd.read_csv(D + "repayment_history.csv", parse_dates=["due_date"])
    rh = rh.sort_values(["customer_id", "loan_id", "due_date"])
    rh["late"] = (rh.max_days_past_due > 0).astype(int)
    rh["seq"] = rh.groupby(["customer_id", "loan_id"]).cumcount()
    prev = rh.groupby(["customer_id", "loan_id"]).late.shift()
    # roll rate: how often a late installment is followed by another late one
    rh["roll"] = rh.late.where(prev == 1)
    rh["cure"] = (1 - rh.late).where(prev == 1)
    run = rh.late.groupby((rh.late != prev).cumsum()).cumsum()
    rh["streak"] = run.where(rh.late == 1, 0)
    g = rh.groupby("customer_id")
    out = pd.DataFrame({"sq_roll_rate": g.roll.mean(), "sq_cure_rate": g.cure.mean(), "sq_max_streak": g.streak.max()})
    # slope of lateness along each loan tells whether the customer is getting worse
    per = rh.groupby(["customer_id", "loan_id"]).apply(
        lambda d: np.polyfit(d.seq, d.max_days_past_due, 1)[0] if len(d) > 2 else np.nan)
    out["sq_dpd_slope_mean"] = per.groupby(level=0).mean()
    out["sq_dpd_slope_max"] = per.groupby(level=0).max()
    last = rh.groupby("customer_id").tail(3).groupby("customer_id")
    out["sq_last3_late"] = last.late.mean()
    out["sq_last3_dpd"] = last.max_days_past_due.mean()
    return out.reset_index()


def category_shift_features(app):
    ec = pd.read_csv(D + "ecommerce_transactions.csv", parse_dates=["transaction_datetime"],
                     usecols=["customer_id", "transaction_datetime", "product_category", "item_total", "order_status"])
    ec = ec[ec.order_status == "completed"].merge(app[["customer_id", "decision_datetime"]], on="customer_id")
    days = (ec.decision_datetime - ec.transaction_datetime).dt.days
    out = []
    for name, m in [("r", days <= 90), ("b", (days > 90) & (days <= 365))]:
        p = ec[m].pivot_table(index="customer_id", columns="product_category", values="item_total", aggfunc="sum", fill_value=0)
        out.append(p.div(p.sum(axis=1), axis=0).add_prefix(f"cs_{name}_"))
    r, b = out
    shift = r.sub(b.rename(columns=lambda c: c.replace("cs_b_", "cs_r_")), fill_value=0)
    res = shift.rename(columns=lambda c: c.replace("cs_r_", "cs_shift_"))
    res["cs_shift_l1"] = shift.abs().sum(axis=1)
    return res.reset_index()


def rhythm_features(app):
    cut = app[["customer_id", "decision_datetime"]].rename(columns={"decision_datetime": "cut"})
    ab = pd.read_csv(D + "account_behavior.csv", parse_dates=["month"]).merge(cut, on="customer_id")
    g = ab[ab.month + pd.offsets.MonthBegin(1) <= ab.cut].groupby("customer_id")
    out = pd.DataFrame({"ab_inc_share": g.income_like_credits.sum() / g.inflows.sum().replace(0, np.nan),
                        "ab_open_avg": g.opening_balance.mean() / g.average_balance.mean().replace(0, np.nan)})
    # the spacing of past applications and of orders says how steady the customer's needs are
    pa = pd.read_csv(D + "previous_applications.csv", parse_dates=["application_datetime"]).merge(cut, on="customer_id")
    pa = pa[pa.application_datetime < pa.cut].sort_values(["customer_id", "application_datetime"])
    pa["gap"] = pa.groupby("customer_id").application_datetime.diff().dt.total_seconds() / 86400
    out = out.join(pa.groupby("customer_id").gap.mean().rename("pa_gap_mean"), how="outer")
    ec = pd.read_csv(D + "ecommerce_transactions.csv", parse_dates=["transaction_datetime"]).merge(cut, on="customer_id")
    od = ec[ec.transaction_datetime < ec.cut].drop_duplicates("order_id").sort_values(["customer_id", "transaction_datetime"])
    od["gap"] = od.groupby("customer_id").transaction_datetime.diff().dt.total_seconds() / 86400
    go = od.groupby("customer_id").gap
    out = out.join(pd.DataFrame({"ec_gap_mean": go.mean(), "ec_gap_last3": go.apply(lambda s: s.tail(3).mean()) / go.mean()}), how="outer")
    return out.reset_index()


def build():
    app = load_apps()
    f = profile_features(app_features(app))
    f = f.merge(bureau_features(app), on="application_id", how="left")
    for fn in [previous_features, repayment_features, account_features, ecommerce_features,
               sequence_features, category_shift_features, rhythm_features]:
        f = f.merge(fn(app), on="customer_id", how="left")

    f = f.copy()
    inc = f.declared_monthly_income
    f["dti_total"] = (f.pa_active_inst.fillna(0) + f.installment) / inc
    f["dti_cashflow"] = (f.pa_active_inst.fillna(0) + f.installment) / f.ab_l3_income_like_credits
    f["inst_to_net"] = f.installment / f.ab_l3_net
    f["income_verified_ratio"] = f.ab_l3_income_like_credits / inc
    f["bureau_debt_to_income"] = f.bu_outstanding_max / (inc * 12)
    f["loan_to_balance"] = f.loan_amount / f.ab_mean_average_balance
    f["rate_vs_history"] = f.interest_rate - f.pa_rate_mean
    f["loan_vs_history"] = f.loan_amount / f.pa_req_mean
    f["x_spend30_to_balance"] = f.ec_spend30 / f.ab_k1_average_balance
    f["x_spend90_to_inflow"] = f.ec_spend90 / (3 * f.ab_l3_inflows)
    f["x_surge_and_drain"] = f.ec_n30_vs_base * f.ab_k1_vs_base_outflows
    f["x_price_and_late"] = f.ec_price_rel * f.rh_rate1_smooth
    f = f.replace([np.inf, -np.inf], np.nan)
    f["target"] = app.target
    return f.drop(columns=["decision_datetime"])


def factor_inputs(app):
    cut = app[["customer_id", "decision_datetime"]].rename(columns={"decision_datetime": "cut"})
    items = {}

    rh = pd.read_csv(D + "repayment_history.csv", parse_dates=["due_date"]).merge(cut, on="customer_id")
    rh["days_ago"] = (rh.cut - rh.due_date).dt.days
    rh = rh[rh.days_ago >= 1].reset_index(drop=True)
    rh["late"] = (rh.max_days_past_due > 0).astype(float)
    rh["ldpd"] = np.log1p(rh.max_days_past_due)
    d = rh.due_date
    rh["f_dow"], rh["f_dom"], rh["f_ym"] = d.dt.dayofweek, d.dt.day // 8, d.dt.to_period("M").astype(str)
    rh["f_seq"] = rh.groupby("loan_id").due_date.rank().clip(upper=12)
    rh["f_term"] = rh.groupby("loan_id").due_date.transform("size").clip(upper=36)
    rh["f_amt"] = pd.qcut(np.log(rh.amount_due), 10, labels=False, duplicates="drop")
    rh["f_age"] = pd.cut(rh.days_ago, [0, 30, 90, 180, 365, 730, 5000], labels=False)
    fx = ["f_dow", "f_dom", "f_ym", "f_seq", "f_term", "f_amt", "f_age"]
    # installment level effects are removed with alternating projections next to a customer effect
    for col in ["late", "ldpd"]:
        eff = {f: np.zeros(len(rh)) for f in fx}
        for _ in range(8):
            cust = (rh[col] - sum(eff.values())).groupby(rh.customer_id).transform("mean").values
            for f in fx:
                rest = rh[col].values - cust - sum(v for k, v in eff.items() if k != f)
                eff[f] = pd.Series(rest).groupby(rh[f].values).transform("mean").values
        items[col] = pd.DataFrame({"c": rh.customer_id, "x": rh[col] - sum(eff.values())})

    ec = pd.read_csv(D + "ecommerce_transactions.csv").merge(cut, on="customer_id")
    ec = ec[ec.quantity > 0]
    u = ec.item_total / ec.quantity
    items["price"] = pd.DataFrame({"c": ec.customer_id, "x": np.log(u / u.groupby(ec.product_category).transform("median"))})

    bu = pd.read_csv(D + "bureau_observations.csv", parse_dates=["available_datetime"]).merge(cut, on="customer_id")
    bu = bu[bu.available_datetime <= bu.cut]
    g = bu.groupby("bureau_provider").score
    bu["z"] = (bu.score - g.transform("mean")) / g.transform("std")
    for c in ["z", "utilization", "inquiries_last_6m", "delinquent_accounts_last_12m"]:
        b = bu[bu[c].notna()]
        items["bu_" + c] = pd.DataFrame({"c": b.customer_id, "x": b[c]})

    ab = pd.read_csv(D + "account_behavior.csv", parse_dates=["month"]).merge(cut, on="customer_id")
    ab = ab[ab.month + pd.offsets.MonthBegin(1) <= ab.cut]
    items["ab_low"] = pd.DataFrame({"c": ab.customer_id, "x": ab.low_balance_days})
    items["ab_outin"] = pd.DataFrame({"c": ab.customer_id, "x": np.log1p(ab.outflows) - np.log1p(ab.inflows)})
    items["ab_ilc"] = pd.DataFrame({"c": ab.customer_id, "x": np.log1p(ab.income_like_credits)})

    pa = pd.read_csv(D + "previous_applications.csv", parse_dates=["decision_datetime"]).merge(cut, on="customer_id")
    pa = pa[pa.decision_datetime < pa.cut]
    items["pa_rej"] = pd.DataFrame({"c": pa.customer_id, "x": (pa.decision == "rejected").astype(float)})
    items["rate"] = pd.DataFrame({"c": app.customer_id, "x": app.annual_interest_rate})

    # each source mean is standardized by its between customer spread, and its noise shrinks with the item count
    M, E = {}, {}
    for j, t in items.items():
        g = t.groupby("c").x
        m, k = g.mean(), g.size()
        w = ((t.x - g.transform("mean")) ** 2).sum() / max(len(t) - len(m), 1)
        b = m.var() - (w / k).mean()
        M[j] = ((m - m.mean()) / np.sqrt(b)).reindex(app.customer_id).values
        E[j] = (w / (k * b)).reindex(app.customer_id).values
    return pd.DataFrame(M), pd.DataFrame(E)


def factor_features(M, E, y, rows):
    # loadings come only from the labelled rows given, and the common scale c is picked by AUC on those rows
    corr = np.array([pd.Series(M[j].values[rows]).corr(pd.Series(y[rows].astype(float))) for j in M.columns])
    best = None
    for c in np.arange(0.30, 0.95, 0.05):
        a = np.clip(corr / c, -0.97, 0.97)
        den = (1 - a ** 2) + E.values
        ok = M.notna().values
        prec = 1 + np.where(ok, a ** 2 / den, 0).sum(1)
        z = np.where(ok, a * M.values / den, 0).sum(1) / prec
        auc = roc_auc_score(y[rows], z[rows])
        if best is None or auc > best[0]:
            best = (auc, z, prec, np.where(ok, M.values * (1 - a ** 2) / den, np.nan))
    out = pd.DataFrame({"factor_z": best[1], "factor_sd": 1 / np.sqrt(best[2])})
    for i, j in enumerate(M.columns):
        if j in ("price", "late", "ldpd", "bu_z"):
            out["fs_" + j] = best[3][:, i]
    return out

### 4.2 `fe_v65.py` — data quality and stress blocks

Missingness and coverage across sources (`dataquality`), bureau direction
(`burdirection`, written but not part of the final frame), and affordability
measured against observed cash flow rather than declared income (`stress`).

In [ ]:
%%writefile fe_v65.py
"""Data-quality, bureau-direction and stress feature blocks.

Three gaps in the existing 554-column feature set, each chosen because it is a
different *kind* of information rather than another statistic on columns that
are already summarised.

`dataquality` — coverage and missingness. In bureau-style credit data, "this
customer has no bureau file" and "this customer's bureau file has no score" are
signals, not gaps to be filled with a mean. The existing blocks encode row
counts, so they capture presence, but not how incomplete the presence is. This
block adds the missingness fractions and the joint-coverage pattern.

`burdirection` — direction, not just level. The existing bureau blocks report
means, maxima and two trend columns for score alone. Nothing encodes whether
utilisation, inquiries or delinquency are *rising*, which is the early-warning
signal a lender would actually act on. This block splits each customer's report
history into an older and a newer half and takes the difference, and adds a
momentum-volatility term that a plain slope cannot express.

`stress` — affordability burden against observed cash flow rather than declared
income. `fe_base.py` has debt-to-income on declared income. Nobody can verify
declared income, while observed inflows and prior active instalments can be
measured. This block recomputes the burden against observed cash flow and
against the instalments the customer is already carrying.
"""

import numpy as np
import pandas as pd

D = "data/"


def _cut(app):
    return app[["application_id", "customer_id", "decision_datetime"]].rename(
        columns={"decision_datetime": "cut"}
    )


def _half_delta(df, key, cols):
    """Difference between the mean over the newer half and the newer half of a
    per-customer report history.

    Splitting by rank rather than by a fixed date keeps the two halves the same
    size even for customers with a thin file, which matters because thin files
    are exactly the ones where a fixed cut-off would leave one half empty.
    """
    df = df.sort_values([key, "available_datetime"])
    out = []
    for _, g in df.groupby(key):
        n = len(g)
        if n < 2:
            for c in cols:
                out.append((g[key].iloc[0], np.nan, c))
            continue
        h = n // 2
        old, new = g.iloc[:h], g.iloc[h:]
        for c in cols:
            out.append((g[key].iloc[0], new[c].mean() - old[c].mean(), c))
    return pd.DataFrame(out, columns=[key, "delta", "col"]).pivot(
        index=key, columns="col", values="delta")


def block_dataquality(app):
    """Coverage depth and missingness fractions across all six tables."""
    cut = _cut(app)[["customer_id", "application_id", "cut"]]
    src = cut.copy()

    bu = pd.read_csv(D + "bureau_observations.csv")
    has_bu = bu.customer_id.nunique()
    m = bu.merge(cut[["customer_id", "cut"]], on="customer_id")
    bu_g = m.groupby("customer_id")
    dq = pd.DataFrame(index=cut.customer_id.unique())
    dq.index.name = "customer_id"
    dq["dq_bureau_score_missing"] = bu_g.score.apply(lambda s: float(s.isna().mean()))
    dq["dq_bureau_util_missing"] = bu_g.utilization.apply(lambda s: float(s.isna().mean()))
    dq["dq_bureau_limit_missing"] = bu_g.revolving_credit_limit.apply(lambda s: float(s.isna().mean()))
    dq["dq_bureau_prov_n"] = bu_g.bureau_provider.nunique()
    dq["dq_bureau_report_n"] = bu_g.size()

    pa = pd.read_csv(D + "previous_applications.csv")
    m = pa.merge(cut[["customer_id", "cut"]], on="customer_id")
    pa_g = m.groupby("customer_id")
    dq["dq_prev_amount_missing"] = pa_g.approved_amount.apply(lambda s: float(s.isna().mean()))
    dq["dq_prev_rate_missing"] = pa_g.annual_interest_rate.apply(lambda s: float(s.isna().mean()))
    dq["dq_prev_no_decision"] = pa_g.decision.apply(lambda s: float(s.isna().mean()))

    rh = pd.read_csv(D + "repayment_history.csv")
    m = rh.merge(cut[["customer_id", "cut"]], on="customer_id")
    rh_g = m.groupby("customer_id")
    dq["dq_repay_dpd_missing"] = rh_g.days_past_due.apply(lambda s: float(s.isna().mean()))
    dq["dq_repay_maxdpd_missing"] = rh_g.max_days_past_due.apply(lambda s: float(s.isna().mean()))

    ab = pd.read_csv(D + "account_behavior.csv")
    m = ab.merge(cut[["customer_id", "cut"]], on="customer_id")
    ab_g = m.groupby("customer_id")
    dq["dq_acct_zero_income_months"] = ab_g.income_like_credits.apply(
        lambda s: float((s.fillna(0) <= 0).mean()))
    dq["dq_acct_balance_missing"] = ab_g.average_balance.apply(lambda s: float(s.isna().mean()))
    dq["dq_acct_months"] = ab_g.size()

    ec = pd.read_csv(D + "ecommerce_transactions.csv")
    m = ec.merge(cut[["customer_id", "cut"]], on="customer_id")
    ec_g = m.groupby("customer_id")
    dq["dq_ec_cat_missing"] = ec_g.product_category.apply(lambda s: float(s.isna().mean()))
    dq["dq_ec_qty_missing"] = ec_g.quantity.apply(lambda s: float(s.isna().mean()))
    dq["dq_ec_item_missing"] = ec_g.item_total.apply(lambda s: float(s.isna().mean()))

    # presence per source, and how many sources agree the customer is visible
    present = pd.DataFrame(index=dq.index)
    idx = present.index
    present["dq_has_bureau"] = (dq.dq_bureau_report_n.fillna(0) > 0).astype(float)
    present["dq_has_prev"] = pd.Series(idx.isin(pa.customer_id.unique()), index=idx).astype(float)
    present["dq_has_repay"] = pd.Series(idx.isin(rh.customer_id.unique()), index=idx).astype(float)
    present["dq_has_acct"] = (dq.dq_acct_months.fillna(0) > 0).astype(float)
    present["dq_has_ec"] = pd.Series(idx.isin(ec.customer_id.unique()), index=idx).astype(float)
    flags = [f"dq_has_{s}" for s in ("bureau", "prev", "repay", "acct", "ec")]
    present["dq_n_sources"] = present[flags].sum(axis=1)
    # visible to the bureau but invisible to our own repayment ledger is a
    # specific contradiction, so keep it separate from a plain count
    present["dq_thin_file"] = (present.dq_n_sources <= 2).astype(float)

    dq = dq.join(present).reset_index()
    out = src[["customer_id", "application_id"]].merge(dq, on="customer_id", how="left")
    return out.drop(columns=["customer_id"])


def block_burdirection(app):
    """Rising versus falling bureau indicators, and momentum volatility."""
    bu = pd.read_csv(D + "bureau_observations.csv",
                     parse_dates=["report_datetime", "available_datetime"])
    cut = _cut(app)[["customer_id", "cut"]]
    m = bu.merge(cut, on="customer_id")
    m["days_ago"] = (m.cut - m.report_datetime).dt.total_seconds() / 86400
    m = m[m.days_ago >= 0]
    key = "customer_id"

    cols = ["score", "utilization", "inquiries_last_6m",
            "delinquent_accounts_last_12m", "outstanding_balance",
            "active_accounts", "revolving_outstanding_balance"]
    delta = _half_delta(m, key, cols)
    delta.columns = ["bd_" + c.replace("_", "") + "_delta" for c in delta.columns]

    # recency-weighted level: a customer whose recent file is worse than their
    # average is a different case from one whose recent file is better
    hl = 180.0
    m["w"] = np.exp(-m.days_ago * np.log(2) / hl)
    g = m.groupby(key)
    wsum = g.w.sum()
    rec = pd.DataFrame(index=delta.index)
    for c in cols:
        num = g.apply(lambda d, c=c: float((d[c] * d.w).sum()), include_groups=False)
        rec["bd_" + c.replace("_", "") + "_rec180"] = (num / wsum).reindex(delta.index)

    # momentum volatility: mean absolute successive change, which a slope
    # cannot express because a slope averages the direction away
    vol = {}
    for c in ["score", "utilization", "inquiries_last_6m", "delinquent_accounts_last_12m"]:
        d = m.sort_values([key, "available_datetime"])
        dif = d.groupby(key)[c].diff().abs()
        vol["bd_" + c.replace("_", "") + "_jump"] = dif.groupby(d[key]).mean()

    gap = (m.sort_values([key, "available_datetime"]).groupby(key).report_datetime
           .apply(lambda s: float((s.diff().dt.total_seconds() / 86400).mean())))
    gap.name = "bd_report_gap_days"

    out = pd.concat([delta, rec, pd.DataFrame(vol), gap.to_frame()], axis=1)

    # a score below 700 is a conventional thin-file boundary; the rate below it
    # is more stable than the raw score because it saturates
    out["bd_score_below_700"] = m.assign(f=(m.score < 700).astype(float)).groupby(key).f.mean()
    out["bd_score_below_600"] = m.assign(f=(m.score < 600).astype(float)).groupby(key).f.mean()
    cmap = app[["customer_id", "application_id"]].drop_duplicates("customer_id")
    return cmap.merge(out.reset_index(), on="customer_id", how="left").drop(columns=["customer_id"])


def block_stress(app):
    """Affordability measured against observed cash and existing commitments."""
    prof = pd.read_csv(D + "customer_profiles.csv")
    cut = _cut(app)
    df = (app.merge(prof, on="customer_id", how="left")
           .merge(cut[["application_id", "cut"]], on="application_id"))

    ab = pd.read_csv(D + "account_behavior.csv")
    m = ab.merge(cut[["customer_id", "cut"]], on="customer_id")
    m["days_ago"] = (m.cut - pd.to_datetime(m.month)).dt.days
    recent = m[(m.days_ago >= 0) & (m.days_ago <= 365)]
    agg = recent.groupby("customer_id").agg(
        af_inflow_last12=("income_like_credits", "mean"),
        af_spend_last12=("outflows", "mean"),
        af_minbal_last12=("minimum_balance", "mean"),
        af_months_last12=("month", "size"),
    )
    base = recent.groupby("customer_id").minimum_balance.apply(
        lambda s: float((s < 1e9).mean()))

    pa = pd.read_csv(D + "previous_applications.csv", parse_dates=["decision_datetime"])
    m = pa.merge(cut[["customer_id", "cut"]], on="customer_id")
    m["days_ago"] = (m.cut - m.decision_datetime).dt.total_seconds() / 86400
    m = m[m.days_ago >= 0]
    appr = m[m.decision == "approved"]
    ap_g = appr.groupby("customer_id")
    burden = ap_g.scheduled_monthly_installment.sum()
    outstanding = ap_g.approved_amount.sum()

    out = df.join(agg, on="customer_id").join(burden.rename("af_prior_inst"), on="customer_id")
    out = out.join(outstanding.rename("af_prior_amount"), on="customer_id")
    out = out.join(base.rename("af_minbal_coverage"), on="customer_id")

    inc = out.declared_monthly_income.replace(0, np.nan)
    inst = out.proposed_monthly_installment
    observed = (out.af_inflow_last12 / out.af_months_last12).replace(0, np.nan)
    annual = inc * 12

    out["af_inst_to_declared"] = inst / inc
    out["af_inst_to_observed"] = inst / observed
    out["af_inst_to_best_estimate"] = inst / observed.fillna(inc)
    # prior commitments are what actually compete for the same cash
    out["af_prior_inst_to_income"] = out.af_prior_inst / inc
    out["af_total_inst_to_income"] = (out.af_prior_inst + inst) / inc
    out["af_prior_amount_to_annual_income"] = out.af_prior_amount / annual
    out["af_total_exposure_to_annual_income"] = (out.af_prior_amount + out.loan_amount) / annual

    hist_max = ap_g.approved_amount.max()
    hist_mean = ap_g.approved_amount.mean()
    out["af_loan_to_hist_max"] = out.loan_amount / hist_max.reindex(out.index)
    out["af_loan_to_hist_mean"] = out.loan_amount / hist_mean.reindex(out.index)

    out["af_interest_burden"] = out.annual_interest_rate * out.loan_amount / annual
    out["af_loan_to_minbal"] = out.loan_amount / out.af_minbal_last12.replace(0, np.nan)
    out["af_observed_minus_declared"] = observed - inc
    out["af_history_years"] = out.af_months_last12 / 12
    # thin observed history next to a large loan is a different risk from a
    # long history, and the ratio separates them where the level does not
    out["af_history_vs_loan"] = out.af_history_years / (out.loan_amount / 1e7)
    # income, age and dependents are already columns in features_cache, and an
    # outer merge on a duplicate name would blank them out
    return out[["application_id"] + [c for c in out.columns if c.startswith("af_")]]

### 4.3 `fe_extra.py` — second-round blocks

Exponentially weighted history of repayment and bureau observations,
within-grade z-scores (`gradenorm`), account and e-commerce extras,
cross-source consistency checks, and a small block of hand-picked interactions.
`build(names)` returns the requested blocks keyed by `application_id`.

In [ ]:
%%writefile fe_extra.py
"""Additional feature blocks that are not part of `fe_base.py`.

Every block is a pure function of the competition tables and the application
decision timestamp, so no label information is used and no fold-specific state
leaks. Nothing here modifies `fe_base.py`.
"""

import sys
from pathlib import Path

import numpy as np
import pandas as pd

D = "data/"
CAT_DISC = ["electronics", "home", "personal_care", "fashion"]
CAT_ESS = ["groceries", "health"]


def _cut(app):
    return app[["application_id", "customer_id", "decision_datetime"]].rename(
        columns={"decision_datetime": "cut"}
    )


def _expw(days_ago, hl):
    return np.exp(-days_ago * np.log(2) / hl)


# ---------------------------------------------------------------- bureau ----
def block_bureau(app):
    bu = pd.read_csv(D + "bureau_observations.csv", parse_dates=["report_datetime", "available_datetime"])
    bu = bu.merge(_cut(app)[["customer_id", "application_id", "cut"]].rename(
        columns={"application_id": "application_id_sel"}), on="customer_id")
    bu["days_ago"] = (bu.cut - bu.report_datetime).dt.total_seconds() / 86400
    bu["avail_ago"] = (bu.cut - bu.available_datetime).dt.total_seconds() / 86400
    bu["prov_lag"] = (bu.available_datetime - bu.report_datetime).dt.total_seconds() / 86400
    bu["delinq_per_acct"] = bu.delinquent_accounts_last_12m / (bu.active_accounts + 1)
    bu["inq_per_acct"] = bu.inquiries_last_6m / (bu.active_accounts + 1)
    bu["out_per_acct"] = bu.outstanding_balance / (bu.active_accounts + 1)
    bu["rev_free"] = bu.revolving_credit_limit - bu.revolving_outstanding_balance

    g = bu.groupby("application_id_sel")
    out = g.agg(
        bx_prov_n=("bureau_provider", "nunique"),
        bx_report_n=("report_datetime", "nunique"),
        bx_avail_ago_min=("avail_ago", "min"),
        bx_avail_ago_mean=("avail_ago", "mean"),
        bx_prov_lag_mean=("prov_lag", "mean"),
        bx_delinq_per_acct_max=("delinq_per_acct", "max"),
        bx_inq_per_acct_max=("inq_per_acct", "max"),
        bx_out_per_acct_max=("out_per_acct", "max"),
        bx_rev_free_min=("rev_free", "min"),
        bx_active_min=("active_accounts", "min"),
        bx_zero_util=("utilization", lambda s: float((s.fillna(0) <= 0).mean())),
    )
    out["bx_score_range"] = g.score.max() - g.score.min()
    out["bx_util_range"] = g.utilization.max() - g.utilization.min()
    out["bx_delinq_range"] = g.delinquent_accounts_last_12m.max() - g.delinquent_accounts_last_12m.min()
    out["bx_inq_range"] = g.inquiries_last_6m.max() - g.inquiries_last_6m.min()

    # provider disagreement: each bureau reads the same customer differently
    z = bu.groupby(["application_id_sel", "bureau_provider"]).score.mean().unstack()
    out["bx_prov_score_std"] = z.std(axis=1)
    out["bx_prov_score_min"] = z.min(axis=1)
    zl = bu.groupby(["application_id_sel", "bureau_provider"]).delinquent_accounts_last_12m.mean().unstack()
    out["bx_prov_delinq_std"] = zl.std(axis=1)

    # short horizon trend: recent reports against everything older
    for col, name in [("score", "bx_score_trend"), ("utilization", "bx_util_trend"),
                      ("delinquent_accounts_last_12m", "bx_delinq_trend"),
                      ("inquiries_last_6m", "bx_inq_trend")]:
        rec = bu[bu.days_ago <= 180].groupby("application_id_sel")[col].mean()
        old = bu[bu.days_ago > 180].groupby("application_id_sel")[col].mean()
        out[name] = (rec - old).reindex(out.index)
        out[name + "_rec"] = rec.reindex(out.index)
    return out.reset_index().rename(columns={"application_id_sel": "application_id"})


# ------------------------------------------------------------- repayment ----
def block_repayment(app):
    rh = pd.read_csv(D + "repayment_history.csv", parse_dates=["due_date", "last_payment_datetime"])
    rh = rh.merge(_cut(app)[["customer_id", "cut"]], on="customer_id")
    rh["days_ago"] = (rh.cut - rh.due_date).dt.total_seconds() / 86400
    rh = rh[rh.days_ago >= 0]
    rh["late"] = (rh.max_days_past_due > 0).astype(int)
    rh["late90"] = (rh.max_days_past_due >= 90).astype(int)
    rh["paid_ratio"] = rh.amount_due.replace(0, np.nan)
    rh["paid_ratio"] = rh.amount_paid / rh.paid_ratio
    rh["pay_delay"] = (rh.last_payment_datetime - rh.due_date).dt.total_seconds() / 86400
    rh["partial"] = ((rh.amount_paid > 0) & (rh.amount_paid < rh.amount_due)).astype(int)
    rh["matured"] = (rh.days_ago >= 30).astype(int)
    # a late instalment that was eventually cleared is a different signal from one that was not
    rh["cured"] = ((rh.max_days_past_due > 0) & (rh.paid_ratio >= 0.95)).astype(int)
    rh["seq"] = rh.groupby(["customer_id", "loan_id"]).cumcount()
    first_late = rh[rh.late == 1].groupby(["customer_id", "loan_id"]).seq.min()
    rh = rh.sort_values(["customer_id", "loan_id", "due_date"])

    g = rh.groupby("customer_id")
    out = g.agg(
        rx_late_recent_days=("days_ago", lambda s: float(s[rh.loc[s.index, "late"].eq(1)].min()) if rh.loc[s.index, "late"].eq(1).any() else np.nan),
        rx_late90_recent_days=("days_ago", lambda s: float(s[rh.loc[s.index, "late90"].eq(1)].min()) if rh.loc[s.index, "late90"].eq(1).any() else np.nan),
        rx_partial_rate=("partial", "mean"),
        rx_matured_rate=("matured", "mean"),
        rx_delay_ge5=("pay_delay", lambda s: float((s > 5).mean())),
        rx_delay_ge15=("pay_delay", lambda s: float((s > 15).mean())),
        rx_delay_ge30=("pay_delay", lambda s: float((s > 30).mean())),
        rx_dpd_mean_matured=("max_days_past_due", "mean"),
    )
    out["rx_cured_of_late"] = rh[rh.late == 1].groupby("customer_id").cured.mean()
    out["rx_late_n"] = g.late.sum()
    out["rx_late90_n"] = g.late90.sum()

    for hl in (60, 180, 365):
        w = _expw(rh.days_ago, hl)
        out[f"rx_late_w{hl}"] = (w * rh.late).groupby(rh.customer_id).sum()
        out[f"rx_dpd_w{hl}"] = (w * rh.max_days_past_due).groupby(rh.customer_id).sum()
        out[f"rx_late_n_w{hl}"] = w.groupby(rh.customer_id).sum()

    per = rh.groupby(["customer_id", "loan_id"]).agg(
        n=("late", "size"), nl=("late", "sum"), mx=("max_days_past_due", "max"),
        end=("due_date", "max"), dpd=("days_past_due", "last"),
    )
    per["late_rate"] = per.nl / per.n
    pg = per.groupby(level=0)
    out["rx_loan_late_rate_mean"] = pg.late_rate.mean()
    out["rx_loan_late_rate_max"] = pg.late_rate.max()
    out["rx_n_loans_late"] = pg.late_rate.apply(lambda s: float((s > 0).mean()))
    out["rx_loan_dpd_max"] = pg.mx.max()
    out["rx_first_late_idx_mean"] = first_late.groupby(level=0).mean()
    out["rx_first_late_idx_min"] = first_late.groupby(level=0).min()
    out["rx_loan_n"] = pg.n.mean()
    out["rx_dpd_open_max"] = pg.dpd.max()
    return out.reset_index()


# ----------------------------------------------------- previous apps --------
def block_previous(app):
    pa = pd.read_csv(D + "previous_applications.csv", parse_dates=["application_datetime", "decision_datetime"])
    pa = pa.merge(_cut(app)[["customer_id", "cut"]], on="customer_id")
    pa["days_ago"] = (pa.cut - pa.application_datetime).dt.total_seconds() / 86400
    pa = pa[pa.days_ago >= 0].sort_values(["customer_id", "application_datetime"])
    pa["rej"] = (pa.decision == "rejected").astype(int)
    pa["can"] = (pa.decision == "cancelled").astype(int)
    pa["app"] = (pa.decision == "approved").astype(int)
    pa["months_elapsed"] = pa.days_ago / 30.44
    pa["active"] = ((pa.decision == "approved") & (pa.months_elapsed < pa.term_months)).astype(int)
    pa["gap"] = pa.groupby("customer_id").application_datetime.diff().dt.total_seconds() / 86400

    g = pa.groupby("customer_id")
    out = pd.DataFrame({
        "px_n30": g.apply(lambda d: float((d.days_ago <= 30).sum()), include_groups=False),
        "px_n90": g.apply(lambda d: float((d.days_ago <= 90).sum()), include_groups=False),
        "px_gap_std": g.gap.std(),
        "px_gap_min": g.gap.min(),
        "px_gap_max": g.gap.max(),
    })
    for flag, name in [("rej", "px_rej"), ("can", "px_can"), ("app", "px_app")]:
        out[f"{name}_recent_days"] = g.apply(
            lambda d: float(d.days_ago[d[flag].eq(1)].min()) if d[flag].eq(1).any() else np.nan,
            include_groups=False)
        out[f"{name}_n90"] = g.apply(
            lambda d: float(((d[flag].eq(1)) & (d.days_ago <= 90)).sum()), include_groups=False)
        out[f"{name}_amt"] = g.apply(
            lambda d: float(d.requested_amount[d[flag].eq(1)].sum()), include_groups=False)
    out["px_active_inst_max"] = g.apply(
        lambda d: float(d.scheduled_monthly_installment[d.active.eq(1)].max()) if d.active.any() else np.nan,
        include_groups=False)
    out["px_active_recent_n"] = g.apply(
        lambda d: float(((d.active.eq(1)) & (d.days_ago <= 365)).sum()), include_groups=False)
    out["px_peak_active_90"] = g.apply(
        lambda d: float(((d.application_datetime > d.application_datetime.max() - pd.Timedelta(days=90)) & d.active.eq(1)).sum()),
        include_groups=False)
    # 90-day rejection bursts: repeated rejections in a short window signal stress
    out["px_rej_burst"] = g.apply(
        lambda d: float(max(
            [((d.rej == 1) & (d.application_datetime >= t) & (d.application_datetime < t + pd.Timedelta(days=90))).sum()
             for t in d.application_datetime], default=0)), include_groups=False)
    return out.reset_index()


# -------------------------------------------------------------- account -----
def block_account(app):
    ab = pd.read_csv(D + "account_behavior.csv", parse_dates=["month"])
    ab = ab.merge(_cut(app)[["customer_id", "cut"]], on="customer_id")
    ab = ab[ab.month + pd.offsets.MonthBegin(1) <= ab.cut].sort_values(["customer_id", "month"])
    ab["net"] = ab.inflows - ab.outflows
    ab["inst"] = ab.customer_id.map(app.set_index("customer_id").proposed_monthly_installment)
    ab["stress"] = (ab.minimum_balance < ab.inst).astype(int)
    ab["cash_stress"] = (ab.income_like_credits < ab.inst).astype(int)
    ab["neg"] = (ab.net < 0).astype(int)

    g = ab.groupby("customer_id")

    def cv(s):
        return float(s.std() / s.abs().mean()) if s.abs().mean() > 0 else np.nan

    out = pd.DataFrame({
        "ax_inc_cv": g.income_like_credits.apply(cv),
        "ax_inflow_cv": g.inflows.apply(cv),
        "ax_outflow_cv": g.outflows.apply(cv),
        "ax_bal_cv": g.average_balance.apply(cv),
        "ax_net_cv": g.net.apply(cv),
        "ax_stress_months": g.stress.sum(),
        "ax_cash_stress_months": g.cash_stress.sum(),
        "ax_neg_months": g.neg.sum(),
        "ax_low_days_mean": g.low_balance_days.mean(),
        "ax_low_days_std": g.low_balance_days.std(),
        "ax_minbal_mean": g.minimum_balance.mean(),
        "ax_close_min": g.closing_balance.min(),
        "ax_max_withdraw_months": g.apply(
            lambda d: float(((d.closing_balance - d.opening_balance) < 0).sum()), include_groups=False),
    })

    def longest_zero(d):
        z = (d.income_like_credits <= 0).to_numpy()
        best = cur = 0
        for v in z:
            cur = cur + 1 if v else 0
            best = max(best, cur)
        return float(best)

    def inc_gap(d):
        v = d.income_like_credits.to_numpy() > 0
        best = 0
        for i in range(1, len(v)):
            if v[i] and v[i - 1]:
                best = max(best, i)
        return float(best)

    def slope(d):
        y = d.income_like_credits.to_numpy(dtype=float)
        n = len(y)
        if n < 3 or y.mean() == 0:
            return np.nan
        x = np.arange(n, dtype=float)
        return float(np.polyfit(x, y, 1)[0] / (abs(y.mean()) + 1e-9) * 12)

    agg = {
        "ax_inc_zero_streak": g.apply(longest_zero, include_groups=False),
        "ax_inc_gap_max": g.apply(inc_gap, include_groups=False),
        "ax_inc_slope": g.apply(slope, include_groups=False),
    }
    out = out.join(pd.DataFrame(agg))

    last3 = ab.groupby("customer_id").tail(3)
    l3 = last3.groupby("customer_id")
    out["ax_inc_last3_min"] = l3.income_like_credits.min()
    out["ax_inc_last3_mean_over_all"] = out.ax_inc_last3_min / g.income_like_credits.mean().replace(0, np.nan)
    out["ax_stress_last3"] = last3.groupby("customer_id").stress.mean()
    out["ax_lowbal_slope"] = out.ax_low_days_mean - g.low_balance_days.mean()
    out["ax_stress_share"] = out.ax_stress_months / g.size()
    return out.reset_index()


# ------------------------------------------------------------ ecommerce -----
def block_ecommerce(app):
    ec = pd.read_csv(
        D + "ecommerce_transactions.csv",
        parse_dates=["transaction_datetime"],
        usecols=["customer_id", "order_id", "transaction_datetime", "product_category",
                 "quantity", "item_total", "order_total", "order_status", "payment_method"],
    )
    ec = ec.merge(_cut(app)[["customer_id", "cut"]], on="customer_id")
    ec = ec[ec.transaction_datetime <= ec.cut]
    ec["days_ago"] = (ec.cut - ec.transaction_datetime).dt.total_seconds() / 86400
    done = ec[ec.order_status == "completed"].copy()
    od = done.drop_duplicates("order_id")

    od["disc"] = od.product_category.isin(CAT_DISC).astype(float)
    g = od.groupby("customer_id")
    out = pd.DataFrame({
        "ex_disc_share": g.disc.mean(),
        "ex_items_per_order": g.quantity.mean(),
        "ex_distinct_items": od.groupby("customer_id").quantity.nunique(),
    })
    out["ex_ess_share"] = g.apply(
        lambda d: float(d.product_category.isin(CAT_ESS).mean()), include_groups=False)
    out["ex_order_max_over_mean"] = g.order_total.max() / g.order_total.mean().replace(0, np.nan)
    out["ex_order_mean_cv"] = g.order_total.apply(
        lambda s: float(s.std() / s.mean()) if s.mean() > 0 else np.nan)
    out["ex_cat_n"] = g.product_category.nunique()
    catshare = od.pivot_table(index="customer_id", columns="product_category",
                              values="order_total", aggfunc="size", fill_value=0)
    share = catshare.div(catshare.sum(axis=1), axis=0)
    out["ex_cat_entropy"] = -(share * np.log(share.replace(0, np.nan))).sum(axis=1)
    out["ex_top_cat_share"] = share.max(axis=1)
    out["ex_electronics_share"] = share.get("electronics", 0.0)

    it = done[done.quantity > 0].copy()
    unit = it.item_total / it.quantity
    it["lrel"] = np.log(unit / unit.groupby(it.product_category).transform("median"))
    gi = it.groupby("customer_id")
    out["ex_lrel_n"] = gi.lrel.size()
    out["ex_lrel_abs_mean"] = gi.lrel.apply(lambda s: float(s.abs().mean()))
    out["ex_lrel_abs_max"] = gi.lrel.apply(lambda s: float(s.abs().max()))
    for hl in (30, 90, 240):
        w = _expw(it.days_ago, hl)
        out[f"ex_lrel_w{hl}"] = (w * it.lrel).groupby(it.customer_id).sum() / w.groupby(it.customer_id).sum()
    out["ex_spend_per_item"] = it.groupby("customer_id").item_total.sum() / it.groupby("customer_id").size()
    return out.reset_index()


# ----------------------------------------------------------- interactions ---
def block_interactions(extra: pd.DataFrame):
    f = extra
    out = pd.DataFrame({"application_id": f.application_id})
    inc = f.declared_monthly_income
    out["ix_rate_x_dti"] = f.interest_rate * f.dti_total
    out["ix_rate_x_income"] = f.interest_rate * np.log1p(inc)
    out["ix_amt_x_age"] = f.loan_amount / (f.age * 1e6 + 1)
    out["ix_inst_x_late"] = f.installment * f.rh_rate1_smooth
    out["ix_rate_x_rent"] = f.interest_rate * (f.housing_status.astype(str) == "rented")
    out["ix_rate_x_self"] = f.interest_rate * (f.employment_type.astype(str) == "self_employed")
    out["ix_rate_x_var"] = f.interest_rate * (f.employment_type.astype(str) == "variable_income")
    out["ix_inst_x_ec_spend"] = f.installment / (f.ec_spend90 / 3 + 1)
    out["ix_inc_vs_spend"] = np.log1p(inc) / np.log1p(f.ec_spend_base / 9 + 1)
    out["ix_rate_x_late"] = f.interest_rate * f.rh_late
    out["ix_rate_x_price"] = f.interest_rate * f.ec_price_rel
    out["ix_dti_x_price"] = f.dti_total * f.ec_price_rel
    out["ix_stress"] = f.installment / (f.ab_l3_income_like_credits + 1)
    return out


BLOCKS = {
    "bureau": block_bureau,
    "repayment": block_repayment,
    "previous": block_previous,
    "account": block_account,
    "ecommerce": block_ecommerce,
    "interactions": block_interactions,
}


# ---------------------------------------------------------------- round 2 ---
GN_KEYS = [
    "ec_price_rel", "ec_price_rel_trend", "ec_spend90_vs_base", "dti_total",
    "loan_to_income", "ab_l3_income_like_credits", "inst_to_income", "bu_score_mean",
    "bu_score_min", "bu_util_max", "bu_delinq_max", "rh_late", "rh_rate1_smooth",
    "rh_maxdpd_max", "pa_reject_rate", "pa_active_n", "ec_orders", "ab_months",
    "declared_monthly_income", "age", "dependents", "employment_tenure_months",
]


def block_gradenorm(app, base=None):
    """Within-grade z-scores.

    The interest rate is a coarse 5 level risk grade. A customer who buys
    expensive electronics is unremarkable inside a high grade bucket but
    remarkable inside a low one, so the same raw feature carries different
    information per grade. Trees cannot express that without a very deep
    interaction, so it is written out explicitly.
    """
    out = pd.DataFrame({"application_id": base.application_id.values})
    for grp in ("interest_rate", "employment_type", "region"):
        g = base[grp].astype(str)
        for c in GN_KEYS:
            if c not in base.columns:
                continue
            v = pd.to_numeric(base[c], errors="coerce")
            m = v.groupby(g).transform("mean")
            s = v.groupby(g).transform("std")
            key = f"gn_{grp[:3]}_{c}"
            out[key] = (v - m) / s.replace(0, np.nan)
    return out


def block_rxtra(app):
    """Affordability history and on-time payment discipline."""
    rh = pd.read_csv(D + "repayment_history.csv", parse_dates=["due_date", "last_payment_datetime"])
    rh = rh.merge(_cut(app)[["customer_id", "cut"]], on="customer_id")
    rh["days_ago"] = (rh.cut - rh.due_date).dt.total_seconds() / 86400
    rh = rh[rh.days_ago >= 0]
    rh["ontime"] = ((rh.max_days_past_due == 0) & (rh.amount_paid >= rh.amount_due)).astype(int)
    rh["matured30"] = (rh.days_ago >= 30).astype(int)
    rh["untouched"] = (rh.max_days_past_due == 0).astype(int)
    per = rh.groupby(["customer_id", "loan_id"]).agg(
        amt=("amount_due", "mean"), mx=("max_days_past_due", "max"),
        n=("amount_due", "size"), end=("due_date", "max"), seq=("due_date", "size"),
    )
    g = per.groupby(level=0)
    out = pd.DataFrame({
        "r2_amt_mean": g.amt.mean(),
        "r2_amt_max": g.amt.max(),
        "r2_amt_min": g.amt.min(),
        "r2_amt_cv": g.amt.apply(lambda s: float(s.std() / s.mean()) if s.mean() > 0 else np.nan),
        "r2_loan_dpd_max": g.mx.max(),
        "r2_loan_dpd_mean": g.mx.mean(),
        "r2_last_amt": per.sort_values("end").groupby(level=0).amt.last(),
    })
    gi = rh.groupby("customer_id")
    out["r2_ontime_rate"] = gi.ontime.sum() / gi.matured30.sum().replace(0, np.nan)
    out["r2_untouched_rate"] = gi.untouched.mean()
    out["r2_amt_slope"] = g.amt.apply(
        lambda s: float(np.polyfit(np.arange(len(s), dtype=float), s.to_numpy(), 1)[0])
        if len(s) > 2 else np.nan)
    out["r2_amt_per_loan_trend"] = out.r2_amt_slope / out.r2_amt_mean.replace(0, np.nan)
    return out.reset_index()


def block_bxtra(app):
    """Depth, span and direction of the bureau file."""
    bu = pd.read_csv(D + "bureau_observations.csv", parse_dates=["report_datetime"])
    bu = bu.merge(_cut(app)[["customer_id", "cut"]], on="customer_id")
    bu = bu[bu.report_datetime <= bu.cut]
    bu["span"] = (bu.cut - bu.report_datetime).dt.total_seconds() / 86400

    def slope(d):
        v = d.score.to_numpy(dtype=float)
        ok = ~np.isnan(v)
        if ok.sum() < 3:
            return np.nan
        x = d.span.to_numpy(dtype=float)[ok]
        return float(np.polyfit(x, v[ok], 1)[0])

    g = bu.groupby("customer_id")
    out = pd.DataFrame({
        "b2_span_max": g.span.max(),
        "b2_months_covered": g.report_datetime.apply(lambda s: s.dt.to_period("M").nunique()),
        "b2_report_gap": g.report_datetime.apply(
            lambda s: float(s.sort_values().diff().dt.total_seconds().median() / 86400)),
        "b2_util_max": g.utilization.max(),
        "b2_util_min": g.utilization.min(),
        "b2_delinq_ever": g.delinquent_accounts_last_12m.max(),
        "b2_active_mean": g.active_accounts.mean(),
        "b2_out_max": g.outstanding_balance.max(),
        "b2_score_slope": g.apply(slope, include_groups=False),
    })
    out["b2_score_span"] = g.span.max() - g.span.min()
    out["b2_thickness"] = g.size() / out.b2_months_covered.replace(0, np.nan)
    first = bu.sort_values("report_datetime").groupby("customer_id").score.first()
    last = bu.sort_values("report_datetime").groupby("customer_id").score.last()
    out["b2_score_first"] = first
    out["b2_score_improve"] = last - first
    return out.reset_index()


def block_extra2(app):
    """Ecommerce tenure, repeat buying and basket composition drift."""
    ec = pd.read_csv(
        D + "ecommerce_transactions.csv",
        parse_dates=["transaction_datetime"],
        usecols=["customer_id", "order_id", "transaction_datetime", "product_category",
                 "quantity", "order_total", "order_status"],
    )
    ec = ec.merge(_cut(app)[["customer_id", "cut"]], on="customer_id")
    ec = ec[ec.transaction_datetime <= ec.cut]
    done = ec[ec.order_status == "completed"].drop_duplicates("order_id").sort_values(
        ["customer_id", "transaction_datetime"])
    done["days_ago"] = (done.cut - done.transaction_datetime).dt.total_seconds() / 86400
    seen = done.groupby("customer_id").product_category.transform("first").notna()
    prior = done.groupby(["customer_id", "product_category"]).cumcount()
    done["repeat_cat"] = (prior > 0).astype(int)

    g = done.groupby("customer_id")
    out = pd.DataFrame({
        "e2_repeat_cat_rate": done.groupby("customer_id").repeat_cat.mean(),
        "e2_qty_std": g.quantity.std(),
        "e2_qty_max": g.quantity.max(),
        "e2_basket_mean": g.order_total.mean(),
        "e2_basket_std": g.order_total.std(),
        "e2_recent_repeat": done[done.days_ago <= 180].groupby("customer_id").repeat_cat.mean(),
    })
    span = (g.transaction_datetime.max() - g.transaction_datetime.min()).dt.total_seconds() / 86400
    out["e2_span_days"] = span
    out["e2_orders_per_month"] = g.size() / (span / 30.44 + 1)
    out["e2_basket_cv"] = out.e2_basket_std / out.e2_basket_mean.replace(0, np.nan)
    firstq = done.groupby("customer_id").quantity.first()
    lastq = done.groupby("customer_id").quantity.last()
    out["e2_qty_trend"] = lastq - firstq
    return out.reset_index()


def block_axtra(app):
    """Monthly cash-flow discipline rather than plain levels."""
    ab = pd.read_csv(D + "account_behavior.csv", parse_dates=["month"])
    ab = ab.merge(_cut(app)[["customer_id", "cut"]], on="customer_id")
    ab = ab[ab.month + pd.offsets.MonthBegin(1) <= ab.cut].sort_values(["customer_id", "month"])
    ab["net"] = ab.inflows - ab.outflows
    ab["disposable"] = ab.income_like_credits - ab.outflows

    g = ab.groupby("customer_id")

    def cv(s):
        return float(s.std() / abs(s.mean())) if abs(s.mean()) > 0 else np.nan

    out = pd.DataFrame({
        "a2_disposable_cv": g.disposable.apply(cv),
        "a2_net_cv": g.net.apply(cv),
        "a2_bal_slope": g.average_balance.apply(
            lambda s: float(np.polyfit(np.arange(len(s), dtype=float), s.to_numpy(), 1)[0])
            if len(s) > 2 else np.nan),
        "a2_min_slope": g.minimum_balance.apply(
            lambda s: float(np.polyfit(np.arange(len(s), dtype=float), s.to_numpy(), 1)[0])
            if len(s) > 2 else np.nan),
        "a2_inc_slope": g.income_like_credits.apply(
            lambda s: float(np.polyfit(np.arange(len(s), dtype=float), s.to_numpy(), 1)[0])
            if len(s) > 2 else np.nan),
    })
    out["a2_net_mean"] = g.net.mean()
    out["a2_net_negative_share"] = ab.groupby("customer_id").net.apply(lambda s: float((s < 0).mean()))
    out["a2_out_gt_inc"] = ab.groupby("customer_id").apply(
        lambda d: float((d.outflows > d.income_like_credits).mean()), include_groups=False)
    out["a2_low_days_worst"] = ab.groupby("customer_id").low_balance_days.max()
    out["a2_low_recent"] = ab.groupby("customer_id").tail(3).groupby("customer_id").low_balance_days.mean()
    out["a2_low_delta"] = out.a2_low_recent - out.a2_low_days_worst
    return out.reset_index()




# ---------------------------------------------------------------- round 3 ---
def block_consistency(app):
    """Cross-source mismatches.

    A customer whose bureau file looks clean while the internal repayment file
    says otherwise is behaving differently from what the bureau reports, and
    the gap itself carries risk information that neither source carries alone.
    The same idea is applied to declared income versus observed cash flow,
    bureau-reported debt versus the instalments we can see, and the size of the
    new instalment versus the largest instalment the customer already carries.
    """
    cp = pd.read_csv(D + "customer_profiles.csv", usecols=["customer_id", "declared_monthly_income"])
    prof = app[["customer_id"]].merge(cp, on="customer_id", how="left")
    inc = prof.set_index("customer_id").declared_monthly_income
    inst = app.set_index("customer_id").proposed_monthly_installment

    pa = pd.read_csv(D + "previous_applications.csv", parse_dates=["application_datetime", "decision_datetime"])
    pa = pa.merge(_cut(app)[["customer_id", "cut"]], on="customer_id")
    pa["days_ago"] = (pa.cut - pa.application_datetime).dt.total_seconds() / 86400
    pa = pa[pa.days_ago >= 0]
    pa["months_elapsed"] = pa.days_ago / 30.44
    pa["active"] = ((pa.decision == "approved") & (pa.months_elapsed < pa.term_months)).astype(float)
    pa["active_inst"] = np.where(pa.active > 0, pa.scheduled_monthly_installment, 0.0)
    pa["remaining"] = np.where(pa.active > 0,
                                pa.approved_amount * (1 - pa.months_elapsed / pa.term_months), 0.0)
    g = pa.groupby("customer_id")
    pa_n_approved = g.apply(lambda d: float((d.decision == "approved").sum()), include_groups=False)
    pa_max_inst = g.active_inst.max()
    pa_inst_sum = g.active_inst.sum()
    pa_recent = pa[pa.days_ago <= 180]
    pa_first_days = g.days_ago.max()

    bu = pd.read_csv(D + "bureau_observations.csv", parse_dates=["report_datetime"])
    bu = bu.merge(_cut(app)[["customer_id", "cut"]], on="customer_id")
    bu = bu[bu.report_datetime <= bu.cut]
    gb = bu.groupby("customer_id")
    bu_score_min = gb.score.min()
    bu_delinq_max = gb.delinquent_accounts_last_12m.max()
    bu_inq_max = gb.inquiries_last_6m.max()
    bu_out_max = gb.outstanding_balance.max()
    bu_active_max = gb.active_accounts.max()
    bu_first_days = gb.report_datetime.apply(lambda s: float((bu.cut.loc[s.index] - s).dt.days.max())
                                             if len(s) else np.nan)

    out = pd.DataFrame(index=app.customer_id)
    yr_income = inc * 12
    out["cx_bureau_vs_declared_debt"] = bu_out_max / (yr_income * 0.35).replace(0, np.nan)
    out["cx_bureau_debt_vs_visible_inst"] = bu_out_max / ((pa_inst_sum + inst) * 12).replace(0, np.nan)
    out["cx_bureau_accounts_vs_our_loans"] = bu_active_max / (pa_n_approved + 1)
    out["cx_inquiries_vs_our_apps"] = bu_inq_max / (pa_n_approved + 1)
    rh = pd.read_csv(D + "repayment_history.csv", parse_dates=["due_date"]).merge(
        _cut(app)[["customer_id", "cut"]], on="customer_id")
    rh["days_ago"] = (rh.cut - rh.due_date).dt.days
    rh = rh[rh.days_ago >= 1]
    rh["bad"] = (rh.max_days_past_due >= 30).astype(float)
    rh_bad_loans = rh.groupby(["customer_id", "loan_id"]).bad.max().groupby("customer_id").sum()
    rh_late_loans = rh[rh.bad > 0].groupby("loan_id").size()
    out["cx_delinq_gap"] = np.log1p(bu_delinq_max.reindex(out.index).fillna(0)) - np.log1p(rh_bad_loans.reindex(out.index).fillna(0))
    out["cx_bureau_delinq_per_internal_late"] = bu_delinq_max / (rh_bad_loans + 1)
    out["cx_file_age_gap_days"] = bu_first_days - pa_first_days
    out["cx_inst_vs_hist_max"] = inst / pa_max_inst.replace(0, np.nan)
    out["cx_inst_vs_hist_mean"] = inst / (pa_max_inst / 2 + 1).replace(0, np.nan)
    out["cx_remaining_vs_inst"] = g.remaining.sum() / (inst * 12).replace(0, np.nan)
    out["cx_recent_app_pressure"] = pa_recent.groupby("customer_id").size()
    out["cx_bureau_known_but_no_apps"] = (bu_score_min.notna()).astype(float) - (pa_n_approved > 0).astype(float)
    return out.reset_index().rename(columns={"index": "customer_id"})


import fe_v65 as _v65  # the v65 blocks, written by the cell above
if _v65 is not None:
    BLOCKS.update({
        "dataquality": _v65.block_dataquality,
        "burdirection": _v65.block_burdirection,
        "stress": _v65.block_stress,
    })


BLOCKS.update({
    "gradenorm": block_gradenorm,
    "rxtra": block_rxtra,
    "bxtra": block_bxtra,
    "ecommerce2": block_extra2,
    "axtra": block_axtra,
    "consistency": block_consistency,
})



BASE_COLS = [
    "declared_monthly_income", "interest_rate", "loan_amount", "installment",
    "housing_status", "employment_type", "dti_total", "rh_rate1_smooth",
    "rh_late", "ec_price_rel", "ec_spend90", "ec_spend_base", "ab_l3_income_like_credits",
    "region", "age", "dependents", "loan_purpose",
]


def build(names=None, app=None, base=None):
    names = list(names) if names else list(BLOCKS)
    if app is None:
        app = pd.read_csv(
            D + "train_applications.csv",
            parse_dates=["application_datetime", "decision_datetime"],
        )
        te = pd.read_csv(D + "test_applications.csv", parse_dates=["application_datetime", "decision_datetime"])
        app = pd.concat([app, te], ignore_index=True)

    cmap = app[["customer_id", "application_id"]].drop_duplicates("customer_id")
    parts = []
    for n in names:
        if n == "interactions":
            continue
        p = BLOCKS[n](app, base=base) if n == "gradenorm" else BLOCKS[n](app)
        if "application_id" not in p.columns:
            p = p.merge(cmap, on="customer_id", how="left").drop(columns=["customer_id"])
        parts.append(p[["application_id"] + [c for c in p.columns if c != "application_id"]])
    if not parts:
        extra = pd.DataFrame({"application_id": app.application_id})
    else:
        extra = parts[0]
    for p in parts[1:]:
        extra = extra.merge(p, on="application_id", how="outer")

    return extra


def with_interactions(extra, base):
    if base is None:
        return extra
    inter = block_interactions(base.merge(extra, on="application_id", how="left"))
    return inter

## 5. Feature frame

`fe_base.build()` gives 311 columns (308 features plus `application_id`,
`customer_id` and `target`). Thirteen second-round blocks and the interaction
block add 290 more, for 598
features in total. Three column sets are then used by the legs:

| set | rule | used by |
|---|---|---|
| `full` | every column | `lgb_full`, `lgbx_full`, `cat_full`, `xgb_full` |
| `lean` | drops demographics and share-type columns with no fold-stable gain | `lgb_lean`, `linear` |
| `fac` | drops the second-order rounding family (`r2_`, `b2_`, `a2_`, `e2_`) | `fac_full` |

### 5.1 Import the modules and define the assembly

Points every module at the data folder and defines `build_feature_frame()`,
which merges the base features with the second-round blocks in a fixed order.
Column order matters: LightGBM samples
columns by position, so a reordered frame would train different trees.

In [ ]:
import fe_base, fe_extra, fe_v65
for module in (fe_base, fe_extra, fe_v65):
    module.D = f"{DATA}/"          # every module reads the CSVs through this prefix

EXTRA_BLOCKS = ["bureau", "repayment", "previous", "account", "ecommerce", "gradenorm",
                "rxtra", "bxtra", "axtra", "ecommerce2", "consistency", "dataquality",
                "stress", "interactions"]


def build_feature_frame():
    """Base features plus the second-round blocks, in the fixed column order."""
    base = fe_base.build()
    app = fe_base.load_apps()
    parts = [fe_extra.build([n], app=app, base=base) for n in EXTRA_BLOCKS]
    parts.append(fe_extra.with_interactions(parts[0], base))
    extra = parts[0]
    for p in parts[1:]:
        extra = extra.merge(p, on="application_id", how="outer")
    add = [c for c in extra.columns if c != "application_id" and c not in base.columns]
    return base.merge(extra[["application_id"] + add], on="application_id", how="left")

### 5.2 Build the frame from the raw CSVs and derive the column sets

Runs `build_feature_frame()` (several minutes), then derives the three column
sets used by the legs.

In [ ]:
t0 = time.time()
frame = build_feature_frame()
print(f"feature frame built from the raw CSVs in {(time.time() - t0) / 60:.1f} min")

NOISE_EXACT = {"region", "employment_type", "housing_status", "age",
               "employment_tenure_months", "dependents", "tenure_to_age",
               "income_per_dependent", "loan_purpose", "decision_lag_h",
               "app_hour", "app_dow"}
cols = [c for c in frame.columns if c not in ("application_id", "customer_id", "target")]
lean = [c for c in cols if c not in NOISE_EXACT
        and not c.startswith(("ec_share_", "ec_catqty_", "cs_shift", "buh_", "gn_"))]
fac = [c for c in cols if not c.startswith(("r2_", "b2_", "a2_", "e2_"))]
print(f"rows x columns    : {frame.shape[0]:,} x {len(cols)}")
print(f"full / lean / fac : {len(cols)} / {len(lean)} / {len(fac)}")
print(f"labelled rows     : {int(frame.target.notna().sum()):,} "
      f"| positives {int(frame.target.sum()):,}")

## 6. Models

Seven legs, each trained with 10-fold stratified cross-validation repeated over
six fold partitions (fold seeds 42, 7, 13, 21, 5, 101). The three plain LightGBM
legs average two booster seeds (0, 1) inside every fold. Nothing is ever fitted
on the test set: test predictions are the mean of the fold models.

| leg | model | columns |
|---|---|---|
| `lgb_full` | LightGBM, 15 leaves, lr 0.01, `lambda_l2` 5, early stopping 300 | `full` |
| `lgb_lean` | same parameters | `lean` |
| `fac_full` | same parameters | `fac` |
| `lgbx_full` | LightGBM, 31 leaves, lr 0.012, `lambda_l2` 12, seed 99 | `full` |
| `cat_full` | CatBoost, depth 6, lr 0.03, `l2_leaf_reg` 5, early stopping 300 | `full` |
| `xgb_full` | XGBoost, depth 4, lr 0.01, `reg_lambda` 20, early stopping 300 | `full` |
| `linear` | quantile transform → spline (4 knots) → logistic regression, C 0.03 | `lean`, numeric only |

The trainer also builds fold-safe factor encodings from each fold's training
labels (`fe_base.factor_inputs` / `factor_features`). They are kept so the code
is unchanged from the run that produced the submission, but none of the seven column lists selects
them, so they do not enter any leg.

### 6.1 `trainer.py` — the seven-leg trainer

`train_legs(frame, fe_base)` loops over the six fold partitions; inside each it
trains every leg on 10 folds and accumulates its out-of-fold predictions (for
the 70,000 training rows) and its averaged test predictions. It returns both for
all seven legs.

In [ ]:
%%writefile trainer.py
"""Seven-leg trainer.

`train_legs(frame, fe_base)` trains every leg with 10-fold cross-validation over
six fold partitions and returns the out-of-fold and test predictions of each.
"""

import time
from types import SimpleNamespace

import lightgbm as lgb
import numpy as np
import pandas as pd
import xgboost as xgb
from catboost import CatBoostClassifier
from scipy.stats import ks_2samp
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import QuantileTransformer, SplineTransformer

NOISE_EXACT = {
    "region", "employment_type", "housing_status", "age", "employment_tenure_months",
    "dependents", "tenure_to_age", "income_per_dependent", "loan_purpose",
    "decision_lag_h", "app_hour", "app_dow",
}
NOISE_PREFIX = ("ec_share_", "ec_catqty_", "cs_shift", "buh_", "gn_")
ROUND2_PREFIX = ("r2_", "b2_", "a2_", "e2_")

LEGS = ["fac_full", "lgb_full", "lgb_lean", "lgbx_full", "cat_full", "xgb_full", "linear"]


def ks_score(y, p):
    return ks_2samp(p[y == 1], p[y == 0]).statistic


def split_frame(f):
    """Train/test design matrices; text columns become categories."""
    tr = f[f.target.notna()].reset_index(drop=True)
    te = f[f.target.isna()].reset_index(drop=True)
    ids = ["application_id", "customer_id", "target"]
    X, Xte = tr.drop(columns=ids), te.drop(columns=ids)
    for c in X.columns:
        if str(X[c].dtype) == "object":
            X[c] = X[c].astype("category")
            Xte[c] = Xte[c].astype("category")
    return X, Xte, tr.application_id.values, te.application_id.values


def train_legs(f, ft, folds=10, fold_seeds="42,7,13,21,5,101", lgb_seeds="0,1",
               lr=0.01, l2=5.0, log=print):
    args = SimpleNamespace(folds=folds, fold_seeds=fold_seeds, lgb_seeds=lgb_seeds,
                           lr=lr, l2=l2)
    X, Xte, train_ids, test_ids = split_frame(f)
    app = ft.load_apps()
    pos = pd.Series(np.arange(len(app)), index=app.application_id)
    p_all = pos[np.concatenate([train_ids, test_ids])].values
    y_app = app.target.values
    n_tr = len(train_ids)
    lgb_seeds = [int(s) for s in args.lgb_seeds.split(",") if s.strip()]
    fold_seeds = [int(s) for s in args.fold_seeds.split(",") if s.strip()]

    full = list(X.columns)
    lean = [c for c in full if c not in NOISE_EXACT and not c.startswith(NOISE_PREFIX)]
    fac_cols = [c for c in full if not c.startswith(ROUND2_PREFIX)]
    log(f"feature counts: full={len(full)} lean={len(lean)} fac={len(fac_cols)}")
    log(f"folds={args.folds} fold_seeds={fold_seeds} lgb_seeds={lgb_seeds}")

    log("computing factor inputs")
    fM, fE = ft.factor_inputs(app)
    log("factor inputs ready")

    def lgb_params(seed, extra=False):
        if extra:
            return dict(objective="binary", learning_rate=args.lr * 1.2, num_leaves=31,
                        min_child_samples=120, feature_fraction=0.35, bagging_fraction=0.85,
                        bagging_freq=1, lambda_l2=args.l2 * 2.4, max_bin=255, seed=seed,
                        deterministic=True, force_row_wise=True, n_jobs=-1, verbose=-1)
        return dict(objective="binary", learning_rate=args.lr, num_leaves=15,
                    min_child_samples=200, feature_fraction=0.3, bagging_fraction=0.8,
                    bagging_freq=1, lambda_l2=args.l2, max_bin=127, seed=seed,
                    deterministic=True, force_row_wise=True, n_jobs=-1, verbose=-1)

    oof_all = {k: np.zeros(n_tr) for k in LEGS}
    test_all = {k: np.zeros(len(Xte)) for k in LEGS}
    seed_oof = {}

    for fs in fold_seeds:
        folds = list(StratifiedKFold(args.folds, shuffle=True,
                                      random_state=fs).split(X, y_app[:n_tr]))
        fac_cols_k = [ft.factor_features(fM, fE, y_app, p_all[a]) for a, _ in folds]
        y = y_app[:n_tr].astype(int)

        cache = {}

        def frame(k):
            if k not in cache:
                fk = fac_cols_k[k]
                cache[k] = (pd.concat([X.reset_index(drop=True),
                                       fk.iloc[p_all[:n_tr]].reset_index(drop=True)], axis=1),
                            pd.concat([Xte.reset_index(drop=True),
                                       fk.iloc[p_all[n_tr:]].reset_index(drop=True)], axis=1))
            return cache[k]

        def run_lgb(cols_by_fold, seed, extra=False):
            oof = np.zeros(n_tr)
            pred = np.zeros(len(Xte))
            for k, (a, b) in enumerate(folds):
                xk, xt = frame(k)
                cols = cols_by_fold[k]
                m = lgb.train(lgb_params(seed, extra), lgb.Dataset(xk.iloc[a][cols], y[a]),
                              num_boost_round=12000 if not extra else 8000,
                              valid_sets=[lgb.Dataset(xk.iloc[b][cols], y[b])],
                              callbacks=[lgb.early_stopping(300, verbose=False)])
                oof[b] = m.predict(xk.iloc[b][cols])
                pred += m.predict(xt[cols]) / len(folds)
            return oof, pred

        def run_cat(cols_by_fold, seed=0):
            oof = np.zeros(n_tr)
            pred = np.zeros(len(Xte))
            for k, (a, b) in enumerate(folds):
                xk, xt = frame(k)
                cols = cols_by_fold[k]
                x = xk[cols].copy()
                xt2 = xt[cols].copy()
                cats = [c for c in cols if str(x[c].dtype) == "category"]
                for c in cats:
                    x[c] = x[c].astype(str)
                    xt2[c] = xt2[c].astype(str)
                m = CatBoostClassifier(iterations=4000, learning_rate=3.0 * args.lr, depth=6,
                                       l2_leaf_reg=5, eval_metric="AUC", od_type="Iter",
                                       od_wait=300, random_seed=seed, cat_features=cats, verbose=0)
                m.fit(x.iloc[a], y[a], eval_set=(x.iloc[b], y[b]))
                oof[b] = m.predict_proba(x.iloc[b])[:, 1]
                pred += m.predict_proba(xt2)[:, 1] / len(folds)
            return oof, pred

        def run_xgb(cols_by_fold, seed=0):
            oof = np.zeros(n_tr)
            pred = np.zeros(len(Xte))
            for k, (a, b) in enumerate(folds):
                xk, xt = frame(k)
                cols = cols_by_fold[k]
                m = xgb.XGBClassifier(n_estimators=9000, learning_rate=args.lr, max_depth=4,
                                      min_child_weight=20, subsample=0.8, colsample_bytree=0.3,
                                      reg_lambda=args.l2 * 4, tree_method="hist", max_bin=128,
                                      enable_categorical=True, eval_metric="auc",
                                      early_stopping_rounds=300, random_state=seed, n_jobs=-1)
                m.fit(xk.iloc[a][cols], y[a], eval_set=[(xk.iloc[b][cols], y[b])], verbose=False)
                oof[b] = m.predict_proba(xk.iloc[b][cols])[:, 1]
                pred += m.predict_proba(xt[cols])[:, 1] / len(folds)
            return oof, pred

        def run_linear(cols):
            num = [c for c in cols if str(X[c].dtype) != "category"]
            # A column that is entirely missing has a NaN median, and fillna with
            # a NaN median leaves the NaN in place, which SplineTransformer
            # rejects. Two v65 `af_` columns are all-NaN, so this is not
            # hypothetical. Dropping them also keeps the linear leg on the same
            # column set v61 used, which is the point of this run.
            num = [c for c in num if X[c].notna().any()]
            flag = [c for c in num if X[c].isna().mean() > 0.01]

            def mat(x, med):
                return pd.concat([x[num].astype(float).fillna(med),
                                  x[flag].isna().astype(float).add_suffix("_na")], axis=1)

            oof = np.zeros(n_tr)
            pred = np.zeros(len(Xte))
            for k, (a, b) in enumerate(folds):
                xk, xt = frame(k)
                med = xk.iloc[a][num].median()
                m = make_pipeline(
                    QuantileTransformer(n_quantiles=200, subsample=10 ** 6, random_state=fs),
                    SplineTransformer(n_knots=4, degree=2),
                    LogisticRegression(C=0.03, max_iter=3000))
                m.fit(mat(xk.iloc[a], med), y[a])
                oof[b] = m.predict_proba(mat(xk.iloc[b], med))[:, 1]
                pred += m.predict_proba(mat(xt, med))[:, 1] / len(folds)
            return oof, pred

        jobs = [
            ("fac_full", [fac_cols] * len(folds), "lgb"),
            ("lgb_full", [full] * len(folds), "lgb"),
            ("lgb_lean", [lean] * len(folds), "lgb"),
            ("lgbx_full", [full] * len(folds), "lgbx"),
            ("cat_full", [full] * len(folds), "cat"),
            ("xgb_full", [full] * len(folds), "xgb"),
            ("linear", lean, "lin"),
        ]
        per_seed = {}
        for key, cols_arg, kind in jobs:
            if kind == "lgb":
                outs = [run_lgb(cols_arg, sd) for sd in lgb_seeds]
                o = np.mean([a for a, _ in outs], axis=0)
                p = np.mean([b for _, b in outs], axis=0)
            else:
                o, p = (run_lgb(cols_arg, 99, True) if kind == "lgbx"
                        else run_cat(cols_arg) if kind == "cat"
                        else run_xgb(cols_arg) if kind == "xgb" else run_linear(cols_arg))
            oof_all[key] += o / len(fold_seeds)
            test_all[key] += p / len(fold_seeds)
            per_seed[key] = float(roc_auc_score(y, o))
            log(f"seed {fs} {key:10s} auc={per_seed[key]:.6f} ks={ks_score(y, o):.6f}")
        seed_oof[fs] = per_seed

    return {"train_ids": train_ids, "test_ids": test_ids,
            **{f"oof_{k}": v for k, v in oof_all.items()},
            **{f"test_{k}": v for k, v in test_all.items()}}

### 6.2 Train the seven legs

Prints one line per leg and fold partition with its out-of-fold AUC and KS (42
lines), also written to `training_log.txt`. The predictions are saved to
`preds_retrained.npz` next to the notebook.

In [ ]:
import trainer

LOG = WORK / "training_log.txt"
LOG.write_text("")


def log(message):
    line = f"{time.strftime('%H:%M:%S')} {message}"
    print(line, flush=True)
    with open(LOG, "a") as fh:
        fh.write(line + "\n")


t0 = time.time()
preds = trainer.train_legs(frame, fe_base, log=log)
np.savez_compressed(WORK / "preds_retrained.npz", **preds)
log(f"training done in {(time.time() - t0) / 3600:.2f} h")

## 7. Blend and submission

KS depends only on the ordering of scores, so every leg is converted to ranks
before blending. The blend of section 7.1 is the submitted recipe.

### 7.1 Blend functions

* `rank01` maps scores to ranks in (0, 1].
* `ks_stat` is the competition metric: the largest gap between the cumulative
  score distributions of defaulted and non-defaulted loans.
* `chain` is the submitted recipe, with every setting exposed as a parameter so
  section 8 can refer to the axes by name:
  `r` = extrapolation ratio, `tm` = linear-leg weight, `cw` = weight of `trees4`
  against `lgbx_full`, `ref` = the leg extrapolated away from.

In [ ]:
def rank01(v):
    return rankdata(v) / len(v)


def ks_stat(score, y):
    order = np.argsort(score, kind="mergesort")
    yy = y[order]
    return float(np.abs(np.cumsum(yy) / yy.sum() - np.cumsum(1 - yy) / (1 - yy).sum()).max())


def chain(d, r=R_FINAL, tm=0.30, cw=2.0, ref="lgb_lean"):
    trees4 = (d["lgb_full"] + d["lgb_lean"] + d["cat_full"] + d["xgb_full"]) / 4
    core = (trees4 * cw + d["lgbx_full"]) / (cw + 1.0)
    tm30 = (1 - tm) * core + tm * d["linear"]
    return rank01(tm30 + r * (tm30 - d[ref]))

### 7.2 Out-of-fold performance of every leg and of the blend

Scores each leg and the submitted blend on the out-of-fold predictions of all
70,000 training applications. These numbers describe the models; section 8
explains why they were not used to pick the blend settings.

In [ ]:
train_ids, test_ids = preds["train_ids"], preds["test_ids"]
y = train.set_index("application_id").loc[train_ids].target.to_numpy().astype(int)
ro = {k[4:]: rank01(v) for k, v in preds.items() if k.startswith("oof_")}
rt = {k[5:]: rank01(v) for k, v in preds.items() if k.startswith("test_")}
oof_final, test_final = chain(ro), chain(rt)

perf = pd.DataFrame({k: (roc_auc_score(y, v), ks_stat(v, y)) for k, v in ro.items()},
                    index=["OOF AUC", "OOF KS"]).T.sort_values("OOF AUC", ascending=False)
perf.loc["SUBMITTED blend"] = (roc_auc_score(y, oof_final), ks_stat(oof_final, y))
print(perf.round(6).to_string())

### 7.3 Write the submission

Writes `submission_DSC0152.csv` next to the notebook.

In [ ]:
OUT_CSV = WORK / "submission_DSC0152.csv"
pd.DataFrame({"application_id": test_ids, "default_probability": test_final}).to_csv(OUT_CSV, index=False)
print(f"wrote {OUT_CSV}")

### 7.4 What the KS statistic measures, on the submitted blend

The KS statistic is the largest vertical gap between the score distributions of
loans that defaulted and loans that did not. The plot shows both cumulative
distributions for the submitted blend on the out-of-fold predictions of all
70,000 training applications, and marks the gap.

In [ ]:
order = np.sort(oof_final)
cdf_pos = np.searchsorted(np.sort(oof_final[y == 1]), order, side="right") / (y == 1).sum()
cdf_neg = np.searchsorted(np.sort(oof_final[y == 0]), order, side="right") / (y == 0).sum()
i = int(np.argmax(cdf_neg - cdf_pos))

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(order, cdf_neg, color="#2a78d6", lw=2, label="no serious delinquency")
ax.plot(order, cdf_pos, color="#eb6834", lw=2, label="reached 90+ DPD")
ax.vlines(order[i], cdf_pos[i], cdf_neg[i], color="#555555", lw=1.5, linestyles="--")
ax.annotate(f"KS = {cdf_neg[i] - cdf_pos[i]:.4f}", (order[i], (cdf_pos[i] + cdf_neg[i]) / 2),
            xytext=(10, 0), textcoords="offset points", va="center", color="#222222")
ax.set_xlabel("blended score (rank, 0 = safest)")
ax.set_ylabel("cumulative share of loans")
ax.set_title("Out-of-fold score distributions of the submitted blend", loc="left")
ax.grid(color="#e6e6e6", lw=0.8)
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False, loc="lower right")
plt.tight_layout()
plt.show()

## 8. How the recipe was chosen

The blend structure was kept because it is the only chain with repeated public
results behind it. What changed last, and what this notebook submits, is the
**extrapolation ratio**: `0.75` instead of the `0.5` used before.

**Why out-of-fold scores were not the selection key.** Two earlier sweeps of
the ratio both picked `0.5` by out-of-fold AUC. On this axis out-of-fold
points the wrong way: `r = 0.75` is −0.000350 AUC and −0.000920 KS against
`r = 0.5`, and +0.00102 on public. The `r = 0.75` vector had been built well
before it was ever submitted, ranked 7th of 9 by an out-of-fold score.

**The ratio axis, mapped.**

| r | public |
|---:|---:|
| 0.50 | 0.56424, 0.56419, 0.56411 (three independent trainings) |
| **0.75** | **0.56520** |
| 1.00 | 0.56464 |

The three `r = 0.50` values come from different feature sets and fold-seed
counts and agree to 0.00013, which is what makes the +0.00102 at `r = 0.75`
meaningful rather than noise.

**What the extrapolation does.** It is a weighted subtraction, not an
amplification:

```
tm30 + r * (tm30 - lgb_lean)  ==  (1 + r) * tm30 - r * lgb_lean
```

so the ratio sets how hard the lean-column leg is subtracted. It amplifies the
residual between the blend and `lgb_lean`, and public rewards that. Why that
residual carries signal is not established.

**Every other setting was moved and lost.** Each axis of the chain was measured
on public on both sides of the submitted value where possible:

| axis | moved to | public | change |
|---|---|---:|---:|
| ratio `r` | 0.50 / 1.00 | 0.56424 / 0.56464 | −0.00096 / −0.00056 |
| linear weight `tm` | 0.20 / 0.40 | 0.56158 / 0.56084 | −0.00362 / −0.00436 |
| core weight | 1:1 / 3:1 | 0.56400 / 0.56294 | −0.00120 / −0.00226 |
| reference leg | `lgb_full` | 0.55934 | −0.00586 |
| `trees4` membership | add `fac_full` | 0.56315 | −0.00205 |
| second core slot | `cat_full` instead of `lgbx_full` | 0.56257 | −0.00263 |
| all three fitted peaks at once | r 0.786, tm 0.295, core 1.85 | 0.56481 | −0.00039 |
| leg quality | three legs retrained, +0.0008 AUC each | 0.56159 | −0.00361 |

Two last probes on the final day — the fitted `r` peak alone, and `cat_full`
removed from `core` — also did not exceed 0.56520, so the recipe above was
frozen. The leg-quality row deserves a plain statement: an 8.7-hour run raised
three legs by +0.00075 to +0.00083 AUC, verified as a controlled A/B, and the
resulting chain scored 0.0036 **lower**. Better legs did not make this chain
better on public.

### 8.1 Public leaderboard ledger

Every scored submission of this chain and its relatives, best first.

In [ ]:
LEDGER = [
    ('r=0.75  tm=0.30  core=2:1  ref=lgb_lean   SUBMITTED', 0.56520),
    ('all three per-axis quadratic peaks at once', 0.56481),
    ('r=1.00', 0.56464),
    ('r=0.50', 0.56424),
    ('r=0.50 on an extended feature set', 0.56419),
    ('r=0.50, ten fold seeds', 0.56411),
    ('core=1:1', 0.56400),
    ('trees4 + fac_full (five trees)', 0.56315),
    ('core=3:1', 0.56294),
    ('second core slot = cat_full', 0.56257),
    ('three legs retrained, +0.0008 AUC each', 0.56159),
    ('tm=0.20', 0.56158),
    ('MLP-heavy blend', 0.56111),
    ('tm=0.40', 0.56084),
    ('80% MLP mix', 0.56073),
    ('MLP + extended-feature pair', 0.56045),
    ('anchor + 5% GRU sequence', 0.55968),
    ('reference leg = lgb_full', 0.55934),
    ('single raw leg', 0.55801),
]
led = pd.DataFrame(LEDGER, columns=["submission", "public"])
led["vs submitted"] = (led.public - 0.56520).round(5)
print(led.to_string(index=False))
print(f"\n{len(led)} scored submissions; the submitted recipe is the maximum.")

## 9. Submission validation

The file must have exactly the columns `application_id,default_probability`,
30,000 rows, the exact test id set, unique ids, and finite probabilities in
[0, 1]. Its SHA-256 is also compared with the file that was submitted to Kaggle.

### 9.1 Run the checks

Each rule is printed as PASS or FAIL; any FAIL stops the notebook.

In [ ]:
sub = pd.read_csv(OUT_CSV)
test_id_set = set(test.application_id)
checks = {
    "columns are application_id,default_probability":
        list(sub.columns) == ["application_id", "default_probability"],
    "30000 rows": len(sub) == 30_000,
    "ids unique": bool(sub.application_id.is_unique),
    "id set equals the test set": set(sub.application_id) == test_id_set,
    "no missing values": not bool(sub.isna().any().any()),
    "all finite": bool(np.isfinite(sub.default_probability).all()),
    "within [0, 1]": bool(sub.default_probability.between(0, 1).all()),
}
digest = hashlib.sha256(OUT_CSV.read_bytes()).hexdigest()
for k, v in checks.items():
    print(f"{'PASS' if v else 'FAIL'}  {k}")
assert all(checks.values()), "submission failed validation"

print(f"\nrows {len(sub):,} | min {sub.default_probability.min():.6f} "
      f"| max {sub.default_probability.max():.6f}")
print(f"sha256 rebuilt   : {digest}")
print(f"sha256 submitted : {SUBMITTED_SHA256}")
print(f"identical to the submitted file: {digest == SUBMITTED_SHA256}")
print("\nSTATUS: PASS")

## 10. Final summary

### 10.1 Result of this run

One table with what this run produced: the mode it ran in, the out-of-fold
quality of the submitted blend, and the file it wrote.

In [ ]:
summary = pd.Series({
    "team": "DSC0152 — kali ini beneran deh",
    "run": "features + training from the raw CSVs",
    "legs blended": len(ro),
    "out-of-fold AUC of the blend": f"{roc_auc_score(y, oof_final):.6f}",
    "out-of-fold KS of the blend": f"{ks_stat(oof_final, y):.6f}",
    "public leaderboard KS": "0.56520",
    "submission file": OUT_CSV.name,
    "rows": f"{len(sub):,}",
    "identical to the submitted file": "yes" if digest == SUBMITTED_SHA256 else "no",
})
print(summary.to_string())

## 11. Limits

* The submitted values are ranks, not calibrated probabilities. KS is invariant
  under any increasing transform, so the ranking is the whole model, but the
  numbers must not be read as default rates.
* `customer_profiles` carries no as-of date, so profile fields are used as
  given. Every other source is truncated at the decision time, which section 3
  re-verifies on every run.
* `fe_base.py` does not apply a positive `days_ago` filter in three of its main
  blocks. That is safe only because the provided tables carry no post-cutoff
  records, which is exactly what section 3 checks.
* The recipe was selected against the public leaderboard, so it carries public
  selection risk: the private split is an independent sample and the measured
  +0.00102 need not reproduce there.
* Training uses fixed seeds and deterministic LightGBM. On a different CPU or
  library version the boosters can still differ in the last bits, so a run
  elsewhere is expected to match the submitted ranking closely but not
  necessarily byte for byte.
* No external data, no AutoML and no automated feature engineering. Every
  feature in sections 4–5 is hand written, and the one tuned blend parameter was
  chosen from an explicitly written grid.